# CNN uLSIF density-ratio estimation

This notebook preserves the original patch-size and convolution experiment
variants. Each large experiment cell is self-contained; run only the section
you intend to reproduce. Generated outputs have been removed for a clean Git
history.

Run the configuration cell first. Override its paths with the
`DRE_PROJECT_ROOT`, `DRE_DATA_ROOT`, `DRE_OUTPUT_ROOT`, `DRE_RASTER_DIR`, or
`DRE_LANDMASK_PATH` environment variables when needed.


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path(os.environ.get("DRE_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
DATA_ROOT = Path(os.environ.get("DRE_DATA_ROOT", PROJECT_ROOT / "data" / "processed")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("DRE_OUTPUT_ROOT", PROJECT_ROOT / "outputs" / "dre_approaches")).expanduser().resolve()
RASTER_DIR = Path(os.environ.get("DRE_RASTER_DIR", DATA_ROOT / "covariate_rasters")).expanduser().resolve()
LANDMASK_PATH = Path(os.environ.get("DRE_LANDMASK_PATH", DATA_ROOT / "landmask.tif")).expanduser().resolve()

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)


# Partial convolution


### Size 3


In [ ]:
# =========================================================
# REAL uLSIF IMPLEMENTATION (Neural uLSIF) — FULL UPDATED CODE
# - replaces BCE classifier-DRE with uLSIF least-squares density-ratio training
# - learns r(x) ≈ p(x)/q(x), where p = y==1, q = y==0
# - saves checkpoints WITHOUT pi0/pi1
# - ensemble outputs ratio (or log-ratio) and bounded score r/(1+r)
# =========================================================

import os
from typing import List, Dict, Tuple, Optional

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_3")
TEST_DIR = str(DATA_ROOT / "test_patches_3")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "ulsif" / "cnn_ulsif_patch_models_3_pconv")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 0.0   # set to 0 for "real uLSIF" (we add explicit L2 lambda below)

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 3  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: pick max Boyce within (loss <= loss_min + delta), optionally with AUC floor
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# uLSIF hyperparameters (this is the important part)
ULSIF_LAMBDA = 1e-3       # explicit L2 penalty (uLSIF lambda)
RATIO_CLIP = 200.0        # clamp ratio for stability (recommended)
CLIP_G = 10.0             # clamp pre-softplus head output (stability)

# For reporting / saving log-ratio
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
# - ENSEMBLE_IN_LOGSPACE=True: geometric mean (weighted avg of log ratios)
# - False: arithmetic mean of ratios
ENSEMBLE_IN_LOGSPACE = False  # for uLSIF squared-risk, arithmetic mean is more aligned


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: ratio -> bounded score, log-ratio
# =========================================================
def ratio_to_bounded_score(r: np.ndarray) -> np.ndarray:
    r = np.asarray(r, dtype=np.float64)
    r = np.clip(r, 0.0, 1e18)
    return r / (1.0 + r)


def ratio_to_logratio(r: np.ndarray) -> np.ndarray:
    r = np.asarray(r, dtype=np.float64)
    r = np.clip(r, 1e-18, 1e18)
    lr = np.log(r)
    lr = np.clip(lr, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return lr


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Data augmentation
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


# =========================================================
# Datasets
# =========================================================
class PatchDREDataset(Dataset):
    """Used only for metric evaluation (mixed labels)."""
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class PatchOnlyDataset(Dataset):
    """Used for uLSIF training: draws from p-only or q-only subsets."""
    def __init__(self, X_values, X_masks, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D. Got {m.shape}")

        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


# =========================================================
# Encoder
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)
        z = self.proj(h)
        return z


# =========================================================
# uLSIF ratio head + model
# =========================================================
class MLPuLSIF(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_g: Optional[float] = 10.0, eps=1e-8):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_g = clip_g
        self.eps = eps

    def forward(self, x):
        g = self.net(x).squeeze(-1)
        if self.clip_g is not None:
            g = torch.clamp(g, -self.clip_g, self.clip_g)
        r = F.softplus(g) + self.eps
        return r


class CNNUlSIF(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_g=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.ratio_head = MLPuLSIF(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_g=clip_g)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        r = self.ratio_head(z)
        return r


# =========================================================
# uLSIF objective + L2 penalty
# =========================================================
def l2_penalty(model: nn.Module) -> torch.Tensor:
    s = torch.zeros([], device=device)
    for p in model.parameters():
        s = s + torch.sum(p * p)
    return s


def ulsif_loss(r_p: torch.Tensor, r_q: torch.Tensor, ratio_clip: Optional[float]) -> torch.Tensor:
    # empirical uLSIF objective: 0.5 E_q[r^2] - E_p[r]
    if ratio_clip is not None:
        r_p = torch.clamp(r_p, 0.0, ratio_clip)
        r_q = torch.clamp(r_q, 0.0, ratio_clip)
    return 0.5 * torch.mean(r_q ** 2) - torch.mean(r_p)


# =========================================================
# Prediction helper: ratio on indices (for metrics / OOF)
# =========================================================
@torch.no_grad()
def predict_ratio_on_indices(
    model: nn.Module,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
) -> np.ndarray:
    ds = PatchDREDataset(X_values, X_masks, y, indices, train=False)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    rs = []
    model.eval()
    for xv, xm, _yb in dl:
        xv, xm = xv.to(device), xm.to(device)
        r = model(xv, xm)
        if RATIO_CLIP is not None:
            r = torch.clamp(r, 0.0, RATIO_CLIP)
        rs.append(r.cpu().numpy())
    return np.concatenate(rs)


# =========================================================
# Training per fold (real uLSIF)
# =========================================================
def train_and_save_fold_model_ulsif(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # define p and q (TRAIN subsets)
    p_tr_idx = train_idx[y_cv[train_idx] == 1]
    q_tr_idx = train_idx[y_cv[train_idx] == 0]
    p_va_idx = val_idx[y_cv[val_idx] == 1]
    q_va_idx = val_idx[y_cv[val_idx] == 0]

    if len(p_tr_idx) == 0 or len(q_tr_idx) == 0:
        raise RuntimeError(f"[fold {fold_id}] Need both p and q in TRAIN. Got p={len(p_tr_idx)} q={len(q_tr_idx)}")

    if len(p_va_idx) == 0 or len(q_va_idx) == 0:
        print(f"[fold {fold_id}] Warning: VAL missing one group. Metrics may be nan. p_val={len(p_va_idx)} q_val={len(q_va_idx)}")

    print(f"\n[fold {fold_id}] train N={len(train_idx)} (p={len(p_tr_idx)}, q={len(q_tr_idx)}), val N={len(val_idx)} (p={len(p_va_idx)}, q={len(q_va_idx)})")

    ds_p_tr = PatchOnlyDataset(X_values, X_masks, p_tr_idx, train=True)
    ds_q_tr = PatchOnlyDataset(X_values, X_masks, q_tr_idx, train=True)
    ds_p_va = PatchOnlyDataset(X_values, X_masks, p_va_idx, train=False)
    ds_q_va = PatchOnlyDataset(X_values, X_masks, q_va_idx, train=False)

    dl_p_tr = DataLoader(ds_p_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, drop_last=False)
    dl_q_tr = DataLoader(ds_q_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, drop_last=False)
    dl_p_va = DataLoader(ds_p_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)
    dl_q_va = DataLoader(ds_q_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNUlSIF(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_g=CLIP_G,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch_ulsif(dl_p, dl_q, opt_obj=None) -> float:
        train = opt_obj is not None
        model.train() if train else model.eval()

        n_steps = 0
        loss_sum = 0.0

        # iterate over the shorter loader each epoch (simple and stable)
        n_batches = min(len(dl_p), len(dl_q))
        it_p = iter(dl_p)
        it_q = iter(dl_q)

        for _ in range(n_batches):
            xv_p, xm_p = next(it_p)
            xv_q, xm_q = next(it_q)
            xv_p, xm_p = xv_p.to(device), xm_p.to(device)
            xv_q, xm_q = xv_q.to(device), xm_q.to(device)

            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                r_p = model(xv_p, xm_p)
                r_q = model(xv_q, xm_q)

                loss = ulsif_loss(r_p, r_q, RATIO_CLIP) + 0.5 * ULSIF_LAMBDA * l2_penalty(model)

                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            loss_sum += float(loss.item())
            n_steps += 1

        return loss_sum / max(1, n_steps)

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr_loss = run_epoch_ulsif(dl_p_tr, dl_q_tr, opt)
        va_loss = run_epoch_ulsif(dl_p_va, dl_q_va, None) if (len(p_va_idx) > 0 and len(q_va_idx) > 0) else np.nan

        # metrics on all val points using bounded score r/(1+r)
        val_r = predict_ratio_on_indices(model, X_values, X_masks, y_cv, val_idx)
        val_score01 = ratio_to_bounded_score(val_r)
        va_mets = compute_boyce_and_auc(y_cv[val_idx], val_score01)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train uLSIF {tr_loss:.4f} | "
            f"val uLSIF {va_loss:.4f} | "
            f"val AUC {va_mets['ROC_AUC']:.4f}, Boyce {va_mets['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va_mets["ROC_AUC"]) and (va_mets["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": int(in_value_channels),
                "patch_size": int(PATCH_SIZE),
                "emb_dim": int(EMB_DIM),
                "hidden_dims": list(HIDDEN_DIMS),
                "ulsif_lambda": float(ULSIF_LAMBDA),
                "ratio_clip": float(RATIO_CLIP) if RATIO_CLIP is not None else None,
                "clip_g": float(CLIP_G) if CLIP_G is not None else None,
            }
            candidates.append(
                {
                    "loss": float(va_loss) if np.isfinite(va_loss) else float(np.inf),
                    "boyce": float(va_mets["Boyce"]) if np.isfinite(va_mets["Boyce"]) else np.nan,
                    "auc": float(va_mets["ROC_AUC"]) if np.isfinite(va_mets["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_r": val_r,
                    "val_y": y_cv[val_idx].copy(),
                    "epoch": ep,
                }
            )

            # early stopping on uLSIF val loss if available; else on train loss
            key_loss = va_loss if np.isfinite(va_loss) else tr_loss
            if key_loss < best_loss_seen - 1e-9:
                best_loss_seen = float(key_loss)
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"val_uLSIF={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded score for this fold's val subset
    val_r = best["val_r"]
    val_score01 = ratio_to_bounded_score(val_r)
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "ulsif_val_loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score + logratio)
# =========================================================
def ensemble_predict_on_indices_ulsif(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != int(in_value_channels):
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != int(PATCH_SIZE):
            raise RuntimeError(f"Patch size mismatch for fold {fid}")

        model = CNNUlSIF(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", EMB_DIM)),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_g=float(state.get("clip_g", CLIP_G)) if state.get("clip_g", None) is not None else None,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)
    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            ratios = []
            for model in models:
                r = model(xv, xm)
                r = torch.clamp(r, min=1e-12, max=RATIO_CLIP if RATIO_CLIP is not None else 1e18)
                ratios.append(r)

            if ENSEMBLE_IN_LOGSPACE:
                log_r = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_r += weights_t[k] * torch.log(ratios[k])
                r_ens = torch.exp(log_r)
            else:
                r_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    r_ens += weights_t[k] * ratios[k]

            r_ens = torch.clamp(r_ens, min=1e-12, max=RATIO_CLIP if RATIO_CLIP is not None else 1e18)
            logratio = torch.log(r_ens)
            logratio = torch.clamp(logratio, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = r_ens / (1.0 + r_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(logratio.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores for comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_ulsif(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["ulsif_val_loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score r/(1+r)):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen uLSIF val losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_ulsif(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score r/(1+r)):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


### Size 5


In [ ]:
# =========================================================
# REAL uLSIF IMPLEMENTATION (Neural uLSIF) — FULL UPDATED CODE
# - replaces BCE classifier-DRE with uLSIF least-squares density-ratio training
# - learns r(x) ≈ p(x)/q(x), where p = y==1, q = y==0
# - saves checkpoints WITHOUT pi0/pi1
# - ensemble outputs ratio (or log-ratio) and bounded score r/(1+r)
# =========================================================

import os
from typing import List, Dict, Tuple, Optional

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_5")
TEST_DIR = str(DATA_ROOT / "test_patches_5")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "ulsif" / "cnn_ulsif_patch_models_5_pconv")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 0.0   # set to 0 for "real uLSIF" (we add explicit L2 lambda below)

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 5  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: pick max Boyce within (loss <= loss_min + delta), optionally with AUC floor
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# uLSIF hyperparameters (this is the important part)
ULSIF_LAMBDA = 1e-3       # explicit L2 penalty (uLSIF lambda)
RATIO_CLIP = 200.0        # clamp ratio for stability (recommended)
CLIP_G = 10.0             # clamp pre-softplus head output (stability)

# For reporting / saving log-ratio
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
# - ENSEMBLE_IN_LOGSPACE=True: geometric mean (weighted avg of log ratios)
# - False: arithmetic mean of ratios
ENSEMBLE_IN_LOGSPACE = False  # for uLSIF squared-risk, arithmetic mean is more aligned


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: ratio -> bounded score, log-ratio
# =========================================================
def ratio_to_bounded_score(r: np.ndarray) -> np.ndarray:
    r = np.asarray(r, dtype=np.float64)
    r = np.clip(r, 0.0, 1e18)
    return r / (1.0 + r)


def ratio_to_logratio(r: np.ndarray) -> np.ndarray:
    r = np.asarray(r, dtype=np.float64)
    r = np.clip(r, 1e-18, 1e18)
    lr = np.log(r)
    lr = np.clip(lr, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return lr


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Data augmentation
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


# =========================================================
# Datasets
# =========================================================
class PatchDREDataset(Dataset):
    """Used only for metric evaluation (mixed labels)."""
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class PatchOnlyDataset(Dataset):
    """Used for uLSIF training: draws from p-only or q-only subsets."""
    def __init__(self, X_values, X_masks, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D. Got {m.shape}")

        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


# =========================================================
# Encoder
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)
        z = self.proj(h)
        return z


# =========================================================
# uLSIF ratio head + model
# =========================================================
class MLPuLSIF(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_g: Optional[float] = 10.0, eps=1e-8):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_g = clip_g
        self.eps = eps

    def forward(self, x):
        g = self.net(x).squeeze(-1)
        if self.clip_g is not None:
            g = torch.clamp(g, -self.clip_g, self.clip_g)
        r = F.softplus(g) + self.eps
        return r


class CNNUlSIF(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_g=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.ratio_head = MLPuLSIF(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_g=clip_g)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        r = self.ratio_head(z)
        return r


# =========================================================
# uLSIF objective + L2 penalty
# =========================================================
def l2_penalty(model: nn.Module) -> torch.Tensor:
    s = torch.zeros([], device=device)
    for p in model.parameters():
        s = s + torch.sum(p * p)
    return s


def ulsif_loss(r_p: torch.Tensor, r_q: torch.Tensor, ratio_clip: Optional[float]) -> torch.Tensor:
    # empirical uLSIF objective: 0.5 E_q[r^2] - E_p[r]
    if ratio_clip is not None:
        r_p = torch.clamp(r_p, 0.0, ratio_clip)
        r_q = torch.clamp(r_q, 0.0, ratio_clip)
    return 0.5 * torch.mean(r_q ** 2) - torch.mean(r_p)


# =========================================================
# Prediction helper: ratio on indices (for metrics / OOF)
# =========================================================
@torch.no_grad()
def predict_ratio_on_indices(
    model: nn.Module,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
) -> np.ndarray:
    ds = PatchDREDataset(X_values, X_masks, y, indices, train=False)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    rs = []
    model.eval()
    for xv, xm, _yb in dl:
        xv, xm = xv.to(device), xm.to(device)
        r = model(xv, xm)
        if RATIO_CLIP is not None:
            r = torch.clamp(r, 0.0, RATIO_CLIP)
        rs.append(r.cpu().numpy())
    return np.concatenate(rs)


# =========================================================
# Training per fold (real uLSIF)
# =========================================================
def train_and_save_fold_model_ulsif(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # define p and q (TRAIN subsets)
    p_tr_idx = train_idx[y_cv[train_idx] == 1]
    q_tr_idx = train_idx[y_cv[train_idx] == 0]
    p_va_idx = val_idx[y_cv[val_idx] == 1]
    q_va_idx = val_idx[y_cv[val_idx] == 0]

    if len(p_tr_idx) == 0 or len(q_tr_idx) == 0:
        raise RuntimeError(f"[fold {fold_id}] Need both p and q in TRAIN. Got p={len(p_tr_idx)} q={len(q_tr_idx)}")

    if len(p_va_idx) == 0 or len(q_va_idx) == 0:
        print(f"[fold {fold_id}] Warning: VAL missing one group. Metrics may be nan. p_val={len(p_va_idx)} q_val={len(q_va_idx)}")

    print(f"\n[fold {fold_id}] train N={len(train_idx)} (p={len(p_tr_idx)}, q={len(q_tr_idx)}), val N={len(val_idx)} (p={len(p_va_idx)}, q={len(q_va_idx)})")

    ds_p_tr = PatchOnlyDataset(X_values, X_masks, p_tr_idx, train=True)
    ds_q_tr = PatchOnlyDataset(X_values, X_masks, q_tr_idx, train=True)
    ds_p_va = PatchOnlyDataset(X_values, X_masks, p_va_idx, train=False)
    ds_q_va = PatchOnlyDataset(X_values, X_masks, q_va_idx, train=False)

    dl_p_tr = DataLoader(ds_p_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, drop_last=False)
    dl_q_tr = DataLoader(ds_q_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, drop_last=False)
    dl_p_va = DataLoader(ds_p_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)
    dl_q_va = DataLoader(ds_q_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNUlSIF(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_g=CLIP_G,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch_ulsif(dl_p, dl_q, opt_obj=None) -> float:
        train = opt_obj is not None
        model.train() if train else model.eval()

        n_steps = 0
        loss_sum = 0.0

        # iterate over the shorter loader each epoch (simple and stable)
        n_batches = min(len(dl_p), len(dl_q))
        it_p = iter(dl_p)
        it_q = iter(dl_q)

        for _ in range(n_batches):
            xv_p, xm_p = next(it_p)
            xv_q, xm_q = next(it_q)
            xv_p, xm_p = xv_p.to(device), xm_p.to(device)
            xv_q, xm_q = xv_q.to(device), xm_q.to(device)

            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                r_p = model(xv_p, xm_p)
                r_q = model(xv_q, xm_q)

                loss = ulsif_loss(r_p, r_q, RATIO_CLIP) + 0.5 * ULSIF_LAMBDA * l2_penalty(model)

                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            loss_sum += float(loss.item())
            n_steps += 1

        return loss_sum / max(1, n_steps)

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr_loss = run_epoch_ulsif(dl_p_tr, dl_q_tr, opt)
        va_loss = run_epoch_ulsif(dl_p_va, dl_q_va, None) if (len(p_va_idx) > 0 and len(q_va_idx) > 0) else np.nan

        # metrics on all val points using bounded score r/(1+r)
        val_r = predict_ratio_on_indices(model, X_values, X_masks, y_cv, val_idx)
        val_score01 = ratio_to_bounded_score(val_r)
        va_mets = compute_boyce_and_auc(y_cv[val_idx], val_score01)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train uLSIF {tr_loss:.4f} | "
            f"val uLSIF {va_loss:.4f} | "
            f"val AUC {va_mets['ROC_AUC']:.4f}, Boyce {va_mets['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va_mets["ROC_AUC"]) and (va_mets["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": int(in_value_channels),
                "patch_size": int(PATCH_SIZE),
                "emb_dim": int(EMB_DIM),
                "hidden_dims": list(HIDDEN_DIMS),
                "ulsif_lambda": float(ULSIF_LAMBDA),
                "ratio_clip": float(RATIO_CLIP) if RATIO_CLIP is not None else None,
                "clip_g": float(CLIP_G) if CLIP_G is not None else None,
            }
            candidates.append(
                {
                    "loss": float(va_loss) if np.isfinite(va_loss) else float(np.inf),
                    "boyce": float(va_mets["Boyce"]) if np.isfinite(va_mets["Boyce"]) else np.nan,
                    "auc": float(va_mets["ROC_AUC"]) if np.isfinite(va_mets["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_r": val_r,
                    "val_y": y_cv[val_idx].copy(),
                    "epoch": ep,
                }
            )

            # early stopping on uLSIF val loss if available; else on train loss
            key_loss = va_loss if np.isfinite(va_loss) else tr_loss
            if key_loss < best_loss_seen - 1e-9:
                best_loss_seen = float(key_loss)
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"val_uLSIF={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded score for this fold's val subset
    val_r = best["val_r"]
    val_score01 = ratio_to_bounded_score(val_r)
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "ulsif_val_loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score + logratio)
# =========================================================
def ensemble_predict_on_indices_ulsif(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != int(in_value_channels):
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != int(PATCH_SIZE):
            raise RuntimeError(f"Patch size mismatch for fold {fid}")

        model = CNNUlSIF(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", EMB_DIM)),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_g=float(state.get("clip_g", CLIP_G)) if state.get("clip_g", None) is not None else None,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)
    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            ratios = []
            for model in models:
                r = model(xv, xm)
                r = torch.clamp(r, min=1e-12, max=RATIO_CLIP if RATIO_CLIP is not None else 1e18)
                ratios.append(r)

            if ENSEMBLE_IN_LOGSPACE:
                log_r = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_r += weights_t[k] * torch.log(ratios[k])
                r_ens = torch.exp(log_r)
            else:
                r_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    r_ens += weights_t[k] * ratios[k]

            r_ens = torch.clamp(r_ens, min=1e-12, max=RATIO_CLIP if RATIO_CLIP is not None else 1e18)
            logratio = torch.log(r_ens)
            logratio = torch.clamp(logratio, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = r_ens / (1.0 + r_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(logratio.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores for comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_ulsif(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["ulsif_val_loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score r/(1+r)):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen uLSIF val losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_ulsif(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score r/(1+r)):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


### Size 13


In [ ]:
# =========================================================
# REAL uLSIF IMPLEMENTATION (Neural uLSIF) — FULL UPDATED CODE
# - replaces BCE classifier-DRE with uLSIF least-squares density-ratio training
# - learns r(x) ≈ p(x)/q(x), where p = y==1, q = y==0
# - saves checkpoints WITHOUT pi0/pi1
# - ensemble outputs ratio (or log-ratio) and bounded score r/(1+r)
# =========================================================

import os
from typing import List, Dict, Tuple, Optional

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_13")
TEST_DIR = str(DATA_ROOT / "test_patches_13")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "ulsif" / "cnn_ulsif_patch_models_13_pconv")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 0.0   # set to 0 for "real uLSIF" (we add explicit L2 lambda below)

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 13  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: pick max Boyce within (loss <= loss_min + delta), optionally with AUC floor
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# uLSIF hyperparameters (this is the important part)
ULSIF_LAMBDA = 1e-3       # explicit L2 penalty (uLSIF lambda)
RATIO_CLIP = 200.0        # clamp ratio for stability (recommended)
CLIP_G = 10.0             # clamp pre-softplus head output (stability)

# For reporting / saving log-ratio
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
# - ENSEMBLE_IN_LOGSPACE=True: geometric mean (weighted avg of log ratios)
# - False: arithmetic mean of ratios
ENSEMBLE_IN_LOGSPACE = False  # for uLSIF squared-risk, arithmetic mean is more aligned


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: ratio -> bounded score, log-ratio
# =========================================================
def ratio_to_bounded_score(r: np.ndarray) -> np.ndarray:
    r = np.asarray(r, dtype=np.float64)
    r = np.clip(r, 0.0, 1e18)
    return r / (1.0 + r)


def ratio_to_logratio(r: np.ndarray) -> np.ndarray:
    r = np.asarray(r, dtype=np.float64)
    r = np.clip(r, 1e-18, 1e18)
    lr = np.log(r)
    lr = np.clip(lr, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return lr


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Data augmentation
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


# =========================================================
# Datasets
# =========================================================
class PatchDREDataset(Dataset):
    """Used only for metric evaluation (mixed labels)."""
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class PatchOnlyDataset(Dataset):
    """Used for uLSIF training: draws from p-only or q-only subsets."""
    def __init__(self, X_values, X_masks, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D. Got {m.shape}")

        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


# =========================================================
# Encoder
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)
        z = self.proj(h)
        return z


# =========================================================
# uLSIF ratio head + model
# =========================================================
class MLPuLSIF(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_g: Optional[float] = 10.0, eps=1e-8):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_g = clip_g
        self.eps = eps

    def forward(self, x):
        g = self.net(x).squeeze(-1)
        if self.clip_g is not None:
            g = torch.clamp(g, -self.clip_g, self.clip_g)
        r = F.softplus(g) + self.eps
        return r


class CNNUlSIF(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_g=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.ratio_head = MLPuLSIF(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_g=clip_g)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        r = self.ratio_head(z)
        return r


# =========================================================
# uLSIF objective + L2 penalty
# =========================================================
def l2_penalty(model: nn.Module) -> torch.Tensor:
    s = torch.zeros([], device=device)
    for p in model.parameters():
        s = s + torch.sum(p * p)
    return s


def ulsif_loss(r_p: torch.Tensor, r_q: torch.Tensor, ratio_clip: Optional[float]) -> torch.Tensor:
    # empirical uLSIF objective: 0.5 E_q[r^2] - E_p[r]
    if ratio_clip is not None:
        r_p = torch.clamp(r_p, 0.0, ratio_clip)
        r_q = torch.clamp(r_q, 0.0, ratio_clip)
    return 0.5 * torch.mean(r_q ** 2) - torch.mean(r_p)


# =========================================================
# Prediction helper: ratio on indices (for metrics / OOF)
# =========================================================
@torch.no_grad()
def predict_ratio_on_indices(
    model: nn.Module,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
) -> np.ndarray:
    ds = PatchDREDataset(X_values, X_masks, y, indices, train=False)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    rs = []
    model.eval()
    for xv, xm, _yb in dl:
        xv, xm = xv.to(device), xm.to(device)
        r = model(xv, xm)
        if RATIO_CLIP is not None:
            r = torch.clamp(r, 0.0, RATIO_CLIP)
        rs.append(r.cpu().numpy())
    return np.concatenate(rs)


# =========================================================
# Training per fold (real uLSIF)
# =========================================================
def train_and_save_fold_model_ulsif(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # define p and q (TRAIN subsets)
    p_tr_idx = train_idx[y_cv[train_idx] == 1]
    q_tr_idx = train_idx[y_cv[train_idx] == 0]
    p_va_idx = val_idx[y_cv[val_idx] == 1]
    q_va_idx = val_idx[y_cv[val_idx] == 0]

    if len(p_tr_idx) == 0 or len(q_tr_idx) == 0:
        raise RuntimeError(f"[fold {fold_id}] Need both p and q in TRAIN. Got p={len(p_tr_idx)} q={len(q_tr_idx)}")

    if len(p_va_idx) == 0 or len(q_va_idx) == 0:
        print(f"[fold {fold_id}] Warning: VAL missing one group. Metrics may be nan. p_val={len(p_va_idx)} q_val={len(q_va_idx)}")

    print(f"\n[fold {fold_id}] train N={len(train_idx)} (p={len(p_tr_idx)}, q={len(q_tr_idx)}), val N={len(val_idx)} (p={len(p_va_idx)}, q={len(q_va_idx)})")

    ds_p_tr = PatchOnlyDataset(X_values, X_masks, p_tr_idx, train=True)
    ds_q_tr = PatchOnlyDataset(X_values, X_masks, q_tr_idx, train=True)
    ds_p_va = PatchOnlyDataset(X_values, X_masks, p_va_idx, train=False)
    ds_q_va = PatchOnlyDataset(X_values, X_masks, q_va_idx, train=False)

    dl_p_tr = DataLoader(ds_p_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, drop_last=False)
    dl_q_tr = DataLoader(ds_q_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, drop_last=False)
    dl_p_va = DataLoader(ds_p_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)
    dl_q_va = DataLoader(ds_q_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNUlSIF(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_g=CLIP_G,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch_ulsif(dl_p, dl_q, opt_obj=None) -> float:
        train = opt_obj is not None
        model.train() if train else model.eval()

        n_steps = 0
        loss_sum = 0.0

        # iterate over the shorter loader each epoch (simple and stable)
        n_batches = min(len(dl_p), len(dl_q))
        it_p = iter(dl_p)
        it_q = iter(dl_q)

        for _ in range(n_batches):
            xv_p, xm_p = next(it_p)
            xv_q, xm_q = next(it_q)
            xv_p, xm_p = xv_p.to(device), xm_p.to(device)
            xv_q, xm_q = xv_q.to(device), xm_q.to(device)

            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                r_p = model(xv_p, xm_p)
                r_q = model(xv_q, xm_q)

                loss = ulsif_loss(r_p, r_q, RATIO_CLIP) + 0.5 * ULSIF_LAMBDA * l2_penalty(model)

                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            loss_sum += float(loss.item())
            n_steps += 1

        return loss_sum / max(1, n_steps)

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr_loss = run_epoch_ulsif(dl_p_tr, dl_q_tr, opt)
        va_loss = run_epoch_ulsif(dl_p_va, dl_q_va, None) if (len(p_va_idx) > 0 and len(q_va_idx) > 0) else np.nan

        # metrics on all val points using bounded score r/(1+r)
        val_r = predict_ratio_on_indices(model, X_values, X_masks, y_cv, val_idx)
        val_score01 = ratio_to_bounded_score(val_r)
        va_mets = compute_boyce_and_auc(y_cv[val_idx], val_score01)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train uLSIF {tr_loss:.4f} | "
            f"val uLSIF {va_loss:.4f} | "
            f"val AUC {va_mets['ROC_AUC']:.4f}, Boyce {va_mets['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va_mets["ROC_AUC"]) and (va_mets["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": int(in_value_channels),
                "patch_size": int(PATCH_SIZE),
                "emb_dim": int(EMB_DIM),
                "hidden_dims": list(HIDDEN_DIMS),
                "ulsif_lambda": float(ULSIF_LAMBDA),
                "ratio_clip": float(RATIO_CLIP) if RATIO_CLIP is not None else None,
                "clip_g": float(CLIP_G) if CLIP_G is not None else None,
            }
            candidates.append(
                {
                    "loss": float(va_loss) if np.isfinite(va_loss) else float(np.inf),
                    "boyce": float(va_mets["Boyce"]) if np.isfinite(va_mets["Boyce"]) else np.nan,
                    "auc": float(va_mets["ROC_AUC"]) if np.isfinite(va_mets["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_r": val_r,
                    "val_y": y_cv[val_idx].copy(),
                    "epoch": ep,
                }
            )

            # early stopping on uLSIF val loss if available; else on train loss
            key_loss = va_loss if np.isfinite(va_loss) else tr_loss
            if key_loss < best_loss_seen - 1e-9:
                best_loss_seen = float(key_loss)
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"val_uLSIF={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded score for this fold's val subset
    val_r = best["val_r"]
    val_score01 = ratio_to_bounded_score(val_r)
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "ulsif_val_loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score + logratio)
# =========================================================
def ensemble_predict_on_indices_ulsif(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != int(in_value_channels):
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != int(PATCH_SIZE):
            raise RuntimeError(f"Patch size mismatch for fold {fid}")

        model = CNNUlSIF(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", EMB_DIM)),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_g=float(state.get("clip_g", CLIP_G)) if state.get("clip_g", None) is not None else None,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)
    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            ratios = []
            for model in models:
                r = model(xv, xm)
                r = torch.clamp(r, min=1e-12, max=RATIO_CLIP if RATIO_CLIP is not None else 1e18)
                ratios.append(r)

            if ENSEMBLE_IN_LOGSPACE:
                log_r = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_r += weights_t[k] * torch.log(ratios[k])
                r_ens = torch.exp(log_r)
            else:
                r_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    r_ens += weights_t[k] * ratios[k]

            r_ens = torch.clamp(r_ens, min=1e-12, max=RATIO_CLIP if RATIO_CLIP is not None else 1e18)
            logratio = torch.log(r_ens)
            logratio = torch.clamp(logratio, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = r_ens / (1.0 + r_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(logratio.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores for comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_ulsif(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["ulsif_val_loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score r/(1+r)):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen uLSIF val losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_ulsif(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score r/(1+r)):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


### Size 23


In [ ]:
# =========================================================
# REAL uLSIF IMPLEMENTATION (Neural uLSIF) — FULL UPDATED CODE
# - replaces BCE classifier-DRE with uLSIF least-squares density-ratio training
# - learns r(x) ≈ p(x)/q(x), where p = y==1, q = y==0
# - saves checkpoints WITHOUT pi0/pi1
# - ensemble outputs ratio (or log-ratio) and bounded score r/(1+r)
# =========================================================

import os
from typing import List, Dict, Tuple, Optional

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_23")
TEST_DIR = str(DATA_ROOT / "test_patches_23")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "ulsif" / "cnn_ulsif_patch_models_23_pconv")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 0.0   # set to 0 for "real uLSIF" (we add explicit L2 lambda below)

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 23  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: pick max Boyce within (loss <= loss_min + delta), optionally with AUC floor
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# uLSIF hyperparameters (this is the important part)
ULSIF_LAMBDA = 1e-3       # explicit L2 penalty (uLSIF lambda)
RATIO_CLIP = 200.0        # clamp ratio for stability (recommended)
CLIP_G = 10.0             # clamp pre-softplus head output (stability)

# For reporting / saving log-ratio
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
# - ENSEMBLE_IN_LOGSPACE=True: geometric mean (weighted avg of log ratios)
# - False: arithmetic mean of ratios
ENSEMBLE_IN_LOGSPACE = False  # for uLSIF squared-risk, arithmetic mean is more aligned


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: ratio -> bounded score, log-ratio
# =========================================================
def ratio_to_bounded_score(r: np.ndarray) -> np.ndarray:
    r = np.asarray(r, dtype=np.float64)
    r = np.clip(r, 0.0, 1e18)
    return r / (1.0 + r)


def ratio_to_logratio(r: np.ndarray) -> np.ndarray:
    r = np.asarray(r, dtype=np.float64)
    r = np.clip(r, 1e-18, 1e18)
    lr = np.log(r)
    lr = np.clip(lr, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return lr


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Data augmentation
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


# =========================================================
# Datasets
# =========================================================
class PatchDREDataset(Dataset):
    """Used only for metric evaluation (mixed labels)."""
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class PatchOnlyDataset(Dataset):
    """Used for uLSIF training: draws from p-only or q-only subsets."""
    def __init__(self, X_values, X_masks, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D. Got {m.shape}")

        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


# =========================================================
# Encoder
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)
        z = self.proj(h)
        return z


# =========================================================
# uLSIF ratio head + model
# =========================================================
class MLPuLSIF(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_g: Optional[float] = 10.0, eps=1e-8):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_g = clip_g
        self.eps = eps

    def forward(self, x):
        g = self.net(x).squeeze(-1)
        if self.clip_g is not None:
            g = torch.clamp(g, -self.clip_g, self.clip_g)
        r = F.softplus(g) + self.eps
        return r


class CNNUlSIF(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_g=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.ratio_head = MLPuLSIF(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_g=clip_g)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        r = self.ratio_head(z)
        return r


# =========================================================
# uLSIF objective + L2 penalty
# =========================================================
def l2_penalty(model: nn.Module) -> torch.Tensor:
    s = torch.zeros([], device=device)
    for p in model.parameters():
        s = s + torch.sum(p * p)
    return s


def ulsif_loss(r_p: torch.Tensor, r_q: torch.Tensor, ratio_clip: Optional[float]) -> torch.Tensor:
    # empirical uLSIF objective: 0.5 E_q[r^2] - E_p[r]
    if ratio_clip is not None:
        r_p = torch.clamp(r_p, 0.0, ratio_clip)
        r_q = torch.clamp(r_q, 0.0, ratio_clip)
    return 0.5 * torch.mean(r_q ** 2) - torch.mean(r_p)


# =========================================================
# Prediction helper: ratio on indices (for metrics / OOF)
# =========================================================
@torch.no_grad()
def predict_ratio_on_indices(
    model: nn.Module,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
) -> np.ndarray:
    ds = PatchDREDataset(X_values, X_masks, y, indices, train=False)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    rs = []
    model.eval()
    for xv, xm, _yb in dl:
        xv, xm = xv.to(device), xm.to(device)
        r = model(xv, xm)
        if RATIO_CLIP is not None:
            r = torch.clamp(r, 0.0, RATIO_CLIP)
        rs.append(r.cpu().numpy())
    return np.concatenate(rs)


# =========================================================
# Training per fold (real uLSIF)
# =========================================================
def train_and_save_fold_model_ulsif(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # define p and q (TRAIN subsets)
    p_tr_idx = train_idx[y_cv[train_idx] == 1]
    q_tr_idx = train_idx[y_cv[train_idx] == 0]
    p_va_idx = val_idx[y_cv[val_idx] == 1]
    q_va_idx = val_idx[y_cv[val_idx] == 0]

    if len(p_tr_idx) == 0 or len(q_tr_idx) == 0:
        raise RuntimeError(f"[fold {fold_id}] Need both p and q in TRAIN. Got p={len(p_tr_idx)} q={len(q_tr_idx)}")

    if len(p_va_idx) == 0 or len(q_va_idx) == 0:
        print(f"[fold {fold_id}] Warning: VAL missing one group. Metrics may be nan. p_val={len(p_va_idx)} q_val={len(q_va_idx)}")

    print(f"\n[fold {fold_id}] train N={len(train_idx)} (p={len(p_tr_idx)}, q={len(q_tr_idx)}), val N={len(val_idx)} (p={len(p_va_idx)}, q={len(q_va_idx)})")

    ds_p_tr = PatchOnlyDataset(X_values, X_masks, p_tr_idx, train=True)
    ds_q_tr = PatchOnlyDataset(X_values, X_masks, q_tr_idx, train=True)
    ds_p_va = PatchOnlyDataset(X_values, X_masks, p_va_idx, train=False)
    ds_q_va = PatchOnlyDataset(X_values, X_masks, q_va_idx, train=False)

    dl_p_tr = DataLoader(ds_p_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, drop_last=False)
    dl_q_tr = DataLoader(ds_q_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, drop_last=False)
    dl_p_va = DataLoader(ds_p_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)
    dl_q_va = DataLoader(ds_q_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNUlSIF(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_g=CLIP_G,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch_ulsif(dl_p, dl_q, opt_obj=None) -> float:
        train = opt_obj is not None
        model.train() if train else model.eval()

        n_steps = 0
        loss_sum = 0.0

        # iterate over the shorter loader each epoch (simple and stable)
        n_batches = min(len(dl_p), len(dl_q))
        it_p = iter(dl_p)
        it_q = iter(dl_q)

        for _ in range(n_batches):
            xv_p, xm_p = next(it_p)
            xv_q, xm_q = next(it_q)
            xv_p, xm_p = xv_p.to(device), xm_p.to(device)
            xv_q, xm_q = xv_q.to(device), xm_q.to(device)

            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                r_p = model(xv_p, xm_p)
                r_q = model(xv_q, xm_q)

                loss = ulsif_loss(r_p, r_q, RATIO_CLIP) + 0.5 * ULSIF_LAMBDA * l2_penalty(model)

                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            loss_sum += float(loss.item())
            n_steps += 1

        return loss_sum / max(1, n_steps)

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr_loss = run_epoch_ulsif(dl_p_tr, dl_q_tr, opt)
        va_loss = run_epoch_ulsif(dl_p_va, dl_q_va, None) if (len(p_va_idx) > 0 and len(q_va_idx) > 0) else np.nan

        # metrics on all val points using bounded score r/(1+r)
        val_r = predict_ratio_on_indices(model, X_values, X_masks, y_cv, val_idx)
        val_score01 = ratio_to_bounded_score(val_r)
        va_mets = compute_boyce_and_auc(y_cv[val_idx], val_score01)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train uLSIF {tr_loss:.4f} | "
            f"val uLSIF {va_loss:.4f} | "
            f"val AUC {va_mets['ROC_AUC']:.4f}, Boyce {va_mets['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va_mets["ROC_AUC"]) and (va_mets["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": int(in_value_channels),
                "patch_size": int(PATCH_SIZE),
                "emb_dim": int(EMB_DIM),
                "hidden_dims": list(HIDDEN_DIMS),
                "ulsif_lambda": float(ULSIF_LAMBDA),
                "ratio_clip": float(RATIO_CLIP) if RATIO_CLIP is not None else None,
                "clip_g": float(CLIP_G) if CLIP_G is not None else None,
            }
            candidates.append(
                {
                    "loss": float(va_loss) if np.isfinite(va_loss) else float(np.inf),
                    "boyce": float(va_mets["Boyce"]) if np.isfinite(va_mets["Boyce"]) else np.nan,
                    "auc": float(va_mets["ROC_AUC"]) if np.isfinite(va_mets["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_r": val_r,
                    "val_y": y_cv[val_idx].copy(),
                    "epoch": ep,
                }
            )

            # early stopping on uLSIF val loss if available; else on train loss
            key_loss = va_loss if np.isfinite(va_loss) else tr_loss
            if key_loss < best_loss_seen - 1e-9:
                best_loss_seen = float(key_loss)
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"val_uLSIF={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded score for this fold's val subset
    val_r = best["val_r"]
    val_score01 = ratio_to_bounded_score(val_r)
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "ulsif_val_loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score + logratio)
# =========================================================
def ensemble_predict_on_indices_ulsif(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != int(in_value_channels):
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != int(PATCH_SIZE):
            raise RuntimeError(f"Patch size mismatch for fold {fid}")

        model = CNNUlSIF(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", EMB_DIM)),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_g=float(state.get("clip_g", CLIP_G)) if state.get("clip_g", None) is not None else None,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)
    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            ratios = []
            for model in models:
                r = model(xv, xm)
                r = torch.clamp(r, min=1e-12, max=RATIO_CLIP if RATIO_CLIP is not None else 1e18)
                ratios.append(r)

            if ENSEMBLE_IN_LOGSPACE:
                log_r = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_r += weights_t[k] * torch.log(ratios[k])
                r_ens = torch.exp(log_r)
            else:
                r_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    r_ens += weights_t[k] * ratios[k]

            r_ens = torch.clamp(r_ens, min=1e-12, max=RATIO_CLIP if RATIO_CLIP is not None else 1e18)
            logratio = torch.log(r_ens)
            logratio = torch.clamp(logratio, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = r_ens / (1.0 + r_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(logratio.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores for comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_ulsif(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["ulsif_val_loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score r/(1+r)):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen uLSIF val losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_ulsif(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score r/(1+r)):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


### Size 33


In [ ]:
# =========================================================
# REAL uLSIF IMPLEMENTATION (Neural uLSIF) — FULL UPDATED CODE
# - replaces BCE classifier-DRE with uLSIF least-squares density-ratio training
# - learns r(x) ≈ p(x)/q(x), where p = y==1, q = y==0
# - saves checkpoints WITHOUT pi0/pi1
# - ensemble outputs ratio (or log-ratio) and bounded score r/(1+r)
# =========================================================

import os
from typing import List, Dict, Tuple, Optional

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_33")
TEST_DIR = str(DATA_ROOT / "test_patches_33")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "ulsif" / "cnn_ulsif_patch_models_33_pconv")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 0.0   # set to 0 for "real uLSIF" (we add explicit L2 lambda below)

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 33  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: pick max Boyce within (loss <= loss_min + delta), optionally with AUC floor
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# uLSIF hyperparameters (this is the important part)
ULSIF_LAMBDA = 1e-3       # explicit L2 penalty (uLSIF lambda)
RATIO_CLIP = 200.0        # clamp ratio for stability (recommended)
CLIP_G = 10.0             # clamp pre-softplus head output (stability)

# For reporting / saving log-ratio
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
# - ENSEMBLE_IN_LOGSPACE=True: geometric mean (weighted avg of log ratios)
# - False: arithmetic mean of ratios
ENSEMBLE_IN_LOGSPACE = False  # for uLSIF squared-risk, arithmetic mean is more aligned


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: ratio -> bounded score, log-ratio
# =========================================================
def ratio_to_bounded_score(r: np.ndarray) -> np.ndarray:
    r = np.asarray(r, dtype=np.float64)
    r = np.clip(r, 0.0, 1e18)
    return r / (1.0 + r)


def ratio_to_logratio(r: np.ndarray) -> np.ndarray:
    r = np.asarray(r, dtype=np.float64)
    r = np.clip(r, 1e-18, 1e18)
    lr = np.log(r)
    lr = np.clip(lr, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return lr


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Data augmentation
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


# =========================================================
# Datasets
# =========================================================
class PatchDREDataset(Dataset):
    """Used only for metric evaluation (mixed labels)."""
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class PatchOnlyDataset(Dataset):
    """Used for uLSIF training: draws from p-only or q-only subsets."""
    def __init__(self, X_values, X_masks, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D. Got {m.shape}")

        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


# =========================================================
# Encoder
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)
        z = self.proj(h)
        return z


# =========================================================
# uLSIF ratio head + model
# =========================================================
class MLPuLSIF(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_g: Optional[float] = 10.0, eps=1e-8):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_g = clip_g
        self.eps = eps

    def forward(self, x):
        g = self.net(x).squeeze(-1)
        if self.clip_g is not None:
            g = torch.clamp(g, -self.clip_g, self.clip_g)
        r = F.softplus(g) + self.eps
        return r


class CNNUlSIF(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_g=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.ratio_head = MLPuLSIF(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_g=clip_g)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        r = self.ratio_head(z)
        return r


# =========================================================
# uLSIF objective + L2 penalty
# =========================================================
def l2_penalty(model: nn.Module) -> torch.Tensor:
    s = torch.zeros([], device=device)
    for p in model.parameters():
        s = s + torch.sum(p * p)
    return s


def ulsif_loss(r_p: torch.Tensor, r_q: torch.Tensor, ratio_clip: Optional[float]) -> torch.Tensor:
    # empirical uLSIF objective: 0.5 E_q[r^2] - E_p[r]
    if ratio_clip is not None:
        r_p = torch.clamp(r_p, 0.0, ratio_clip)
        r_q = torch.clamp(r_q, 0.0, ratio_clip)
    return 0.5 * torch.mean(r_q ** 2) - torch.mean(r_p)


# =========================================================
# Prediction helper: ratio on indices (for metrics / OOF)
# =========================================================
@torch.no_grad()
def predict_ratio_on_indices(
    model: nn.Module,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
) -> np.ndarray:
    ds = PatchDREDataset(X_values, X_masks, y, indices, train=False)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    rs = []
    model.eval()
    for xv, xm, _yb in dl:
        xv, xm = xv.to(device), xm.to(device)
        r = model(xv, xm)
        if RATIO_CLIP is not None:
            r = torch.clamp(r, 0.0, RATIO_CLIP)
        rs.append(r.cpu().numpy())
    return np.concatenate(rs)


# =========================================================
# Training per fold (real uLSIF)
# =========================================================
def train_and_save_fold_model_ulsif(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # define p and q (TRAIN subsets)
    p_tr_idx = train_idx[y_cv[train_idx] == 1]
    q_tr_idx = train_idx[y_cv[train_idx] == 0]
    p_va_idx = val_idx[y_cv[val_idx] == 1]
    q_va_idx = val_idx[y_cv[val_idx] == 0]

    if len(p_tr_idx) == 0 or len(q_tr_idx) == 0:
        raise RuntimeError(f"[fold {fold_id}] Need both p and q in TRAIN. Got p={len(p_tr_idx)} q={len(q_tr_idx)}")

    if len(p_va_idx) == 0 or len(q_va_idx) == 0:
        print(f"[fold {fold_id}] Warning: VAL missing one group. Metrics may be nan. p_val={len(p_va_idx)} q_val={len(q_va_idx)}")

    print(f"\n[fold {fold_id}] train N={len(train_idx)} (p={len(p_tr_idx)}, q={len(q_tr_idx)}), val N={len(val_idx)} (p={len(p_va_idx)}, q={len(q_va_idx)})")

    ds_p_tr = PatchOnlyDataset(X_values, X_masks, p_tr_idx, train=True)
    ds_q_tr = PatchOnlyDataset(X_values, X_masks, q_tr_idx, train=True)
    ds_p_va = PatchOnlyDataset(X_values, X_masks, p_va_idx, train=False)
    ds_q_va = PatchOnlyDataset(X_values, X_masks, q_va_idx, train=False)

    dl_p_tr = DataLoader(ds_p_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, drop_last=False)
    dl_q_tr = DataLoader(ds_q_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, drop_last=False)
    dl_p_va = DataLoader(ds_p_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)
    dl_q_va = DataLoader(ds_q_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNUlSIF(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_g=CLIP_G,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch_ulsif(dl_p, dl_q, opt_obj=None) -> float:
        train = opt_obj is not None
        model.train() if train else model.eval()

        n_steps = 0
        loss_sum = 0.0

        # iterate over the shorter loader each epoch (simple and stable)
        n_batches = min(len(dl_p), len(dl_q))
        it_p = iter(dl_p)
        it_q = iter(dl_q)

        for _ in range(n_batches):
            xv_p, xm_p = next(it_p)
            xv_q, xm_q = next(it_q)
            xv_p, xm_p = xv_p.to(device), xm_p.to(device)
            xv_q, xm_q = xv_q.to(device), xm_q.to(device)

            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                r_p = model(xv_p, xm_p)
                r_q = model(xv_q, xm_q)

                loss = ulsif_loss(r_p, r_q, RATIO_CLIP) + 0.5 * ULSIF_LAMBDA * l2_penalty(model)

                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            loss_sum += float(loss.item())
            n_steps += 1

        return loss_sum / max(1, n_steps)

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr_loss = run_epoch_ulsif(dl_p_tr, dl_q_tr, opt)
        va_loss = run_epoch_ulsif(dl_p_va, dl_q_va, None) if (len(p_va_idx) > 0 and len(q_va_idx) > 0) else np.nan

        # metrics on all val points using bounded score r/(1+r)
        val_r = predict_ratio_on_indices(model, X_values, X_masks, y_cv, val_idx)
        val_score01 = ratio_to_bounded_score(val_r)
        va_mets = compute_boyce_and_auc(y_cv[val_idx], val_score01)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train uLSIF {tr_loss:.4f} | "
            f"val uLSIF {va_loss:.4f} | "
            f"val AUC {va_mets['ROC_AUC']:.4f}, Boyce {va_mets['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va_mets["ROC_AUC"]) and (va_mets["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": int(in_value_channels),
                "patch_size": int(PATCH_SIZE),
                "emb_dim": int(EMB_DIM),
                "hidden_dims": list(HIDDEN_DIMS),
                "ulsif_lambda": float(ULSIF_LAMBDA),
                "ratio_clip": float(RATIO_CLIP) if RATIO_CLIP is not None else None,
                "clip_g": float(CLIP_G) if CLIP_G is not None else None,
            }
            candidates.append(
                {
                    "loss": float(va_loss) if np.isfinite(va_loss) else float(np.inf),
                    "boyce": float(va_mets["Boyce"]) if np.isfinite(va_mets["Boyce"]) else np.nan,
                    "auc": float(va_mets["ROC_AUC"]) if np.isfinite(va_mets["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_r": val_r,
                    "val_y": y_cv[val_idx].copy(),
                    "epoch": ep,
                }
            )

            # early stopping on uLSIF val loss if available; else on train loss
            key_loss = va_loss if np.isfinite(va_loss) else tr_loss
            if key_loss < best_loss_seen - 1e-9:
                best_loss_seen = float(key_loss)
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"val_uLSIF={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded score for this fold's val subset
    val_r = best["val_r"]
    val_score01 = ratio_to_bounded_score(val_r)
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "ulsif_val_loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score + logratio)
# =========================================================
def ensemble_predict_on_indices_ulsif(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != int(in_value_channels):
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != int(PATCH_SIZE):
            raise RuntimeError(f"Patch size mismatch for fold {fid}")

        model = CNNUlSIF(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", EMB_DIM)),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_g=float(state.get("clip_g", CLIP_G)) if state.get("clip_g", None) is not None else None,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)
    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            ratios = []
            for model in models:
                r = model(xv, xm)
                r = torch.clamp(r, min=1e-12, max=RATIO_CLIP if RATIO_CLIP is not None else 1e18)
                ratios.append(r)

            if ENSEMBLE_IN_LOGSPACE:
                log_r = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_r += weights_t[k] * torch.log(ratios[k])
                r_ens = torch.exp(log_r)
            else:
                r_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    r_ens += weights_t[k] * ratios[k]

            r_ens = torch.clamp(r_ens, min=1e-12, max=RATIO_CLIP if RATIO_CLIP is not None else 1e18)
            logratio = torch.log(r_ens)
            logratio = torch.clamp(logratio, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = r_ens / (1.0 + r_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(logratio.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores for comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_ulsif(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["ulsif_val_loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score r/(1+r)):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen uLSIF val losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_ulsif(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score r/(1+r)):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


### Size 65


In [ ]:
# =========================================================
# REAL uLSIF IMPLEMENTATION (Neural uLSIF) — FULL UPDATED CODE
# - replaces BCE classifier-DRE with uLSIF least-squares density-ratio training
# - learns r(x) ≈ p(x)/q(x), where p = y==1, q = y==0
# - saves checkpoints WITHOUT pi0/pi1
# - ensemble outputs ratio (or log-ratio) and bounded score r/(1+r)
# =========================================================

import os
from typing import List, Dict, Tuple, Optional

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_65")
TEST_DIR = str(DATA_ROOT / "test_patches_65")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "ulsif" / "cnn_ulsif_patch_models_65_pconv")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 0.0   # set to 0 for "real uLSIF" (we add explicit L2 lambda below)

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 65  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: pick max Boyce within (loss <= loss_min + delta), optionally with AUC floor
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# uLSIF hyperparameters (this is the important part)
ULSIF_LAMBDA = 1e-3       # explicit L2 penalty (uLSIF lambda)
RATIO_CLIP = 200.0        # clamp ratio for stability (recommended)
CLIP_G = 10.0             # clamp pre-softplus head output (stability)

# For reporting / saving log-ratio
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
# - ENSEMBLE_IN_LOGSPACE=True: geometric mean (weighted avg of log ratios)
# - False: arithmetic mean of ratios
ENSEMBLE_IN_LOGSPACE = False  # for uLSIF squared-risk, arithmetic mean is more aligned


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: ratio -> bounded score, log-ratio
# =========================================================
def ratio_to_bounded_score(r: np.ndarray) -> np.ndarray:
    r = np.asarray(r, dtype=np.float64)
    r = np.clip(r, 0.0, 1e18)
    return r / (1.0 + r)


def ratio_to_logratio(r: np.ndarray) -> np.ndarray:
    r = np.asarray(r, dtype=np.float64)
    r = np.clip(r, 1e-18, 1e18)
    lr = np.log(r)
    lr = np.clip(lr, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return lr


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Data augmentation
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


# =========================================================
# Datasets
# =========================================================
class PatchDREDataset(Dataset):
    """Used only for metric evaluation (mixed labels)."""
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class PatchOnlyDataset(Dataset):
    """Used for uLSIF training: draws from p-only or q-only subsets."""
    def __init__(self, X_values, X_masks, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D. Got {m.shape}")

        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


# =========================================================
# Encoder
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(nn.Linear(32, emb_dim), nn.ReLU(inplace=True))

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)
        z = self.proj(h)
        return z


# =========================================================
# uLSIF ratio head + model
# =========================================================
class MLPuLSIF(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_g: Optional[float] = 10.0, eps=1e-8):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_g = clip_g
        self.eps = eps

    def forward(self, x):
        g = self.net(x).squeeze(-1)
        if self.clip_g is not None:
            g = torch.clamp(g, -self.clip_g, self.clip_g)
        r = F.softplus(g) + self.eps
        return r


class CNNUlSIF(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_g=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.ratio_head = MLPuLSIF(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_g=clip_g)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        r = self.ratio_head(z)
        return r


# =========================================================
# uLSIF objective + L2 penalty
# =========================================================
def l2_penalty(model: nn.Module) -> torch.Tensor:
    s = torch.zeros([], device=device)
    for p in model.parameters():
        s = s + torch.sum(p * p)
    return s


def ulsif_loss(r_p: torch.Tensor, r_q: torch.Tensor, ratio_clip: Optional[float]) -> torch.Tensor:
    # empirical uLSIF objective: 0.5 E_q[r^2] - E_p[r]
    if ratio_clip is not None:
        r_p = torch.clamp(r_p, 0.0, ratio_clip)
        r_q = torch.clamp(r_q, 0.0, ratio_clip)
    return 0.5 * torch.mean(r_q ** 2) - torch.mean(r_p)


# =========================================================
# Prediction helper: ratio on indices (for metrics / OOF)
# =========================================================
@torch.no_grad()
def predict_ratio_on_indices(
    model: nn.Module,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
) -> np.ndarray:
    ds = PatchDREDataset(X_values, X_masks, y, indices, train=False)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    rs = []
    model.eval()
    for xv, xm, _yb in dl:
        xv, xm = xv.to(device), xm.to(device)
        r = model(xv, xm)
        if RATIO_CLIP is not None:
            r = torch.clamp(r, 0.0, RATIO_CLIP)
        rs.append(r.cpu().numpy())
    return np.concatenate(rs)


# =========================================================
# Training per fold (real uLSIF)
# =========================================================
def train_and_save_fold_model_ulsif(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # define p and q (TRAIN subsets)
    p_tr_idx = train_idx[y_cv[train_idx] == 1]
    q_tr_idx = train_idx[y_cv[train_idx] == 0]
    p_va_idx = val_idx[y_cv[val_idx] == 1]
    q_va_idx = val_idx[y_cv[val_idx] == 0]

    if len(p_tr_idx) == 0 or len(q_tr_idx) == 0:
        raise RuntimeError(f"[fold {fold_id}] Need both p and q in TRAIN. Got p={len(p_tr_idx)} q={len(q_tr_idx)}")

    if len(p_va_idx) == 0 or len(q_va_idx) == 0:
        print(f"[fold {fold_id}] Warning: VAL missing one group. Metrics may be nan. p_val={len(p_va_idx)} q_val={len(q_va_idx)}")

    print(f"\n[fold {fold_id}] train N={len(train_idx)} (p={len(p_tr_idx)}, q={len(q_tr_idx)}), val N={len(val_idx)} (p={len(p_va_idx)}, q={len(q_va_idx)})")

    ds_p_tr = PatchOnlyDataset(X_values, X_masks, p_tr_idx, train=True)
    ds_q_tr = PatchOnlyDataset(X_values, X_masks, q_tr_idx, train=True)
    ds_p_va = PatchOnlyDataset(X_values, X_masks, p_va_idx, train=False)
    ds_q_va = PatchOnlyDataset(X_values, X_masks, q_va_idx, train=False)

    dl_p_tr = DataLoader(ds_p_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, drop_last=False)
    dl_q_tr = DataLoader(ds_q_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, drop_last=False)
    dl_p_va = DataLoader(ds_p_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)
    dl_q_va = DataLoader(ds_q_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNUlSIF(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_g=CLIP_G,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch_ulsif(dl_p, dl_q, opt_obj=None) -> float:
        train = opt_obj is not None
        model.train() if train else model.eval()

        n_steps = 0
        loss_sum = 0.0

        # iterate over the shorter loader each epoch (simple and stable)
        n_batches = min(len(dl_p), len(dl_q))
        it_p = iter(dl_p)
        it_q = iter(dl_q)

        for _ in range(n_batches):
            xv_p, xm_p = next(it_p)
            xv_q, xm_q = next(it_q)
            xv_p, xm_p = xv_p.to(device), xm_p.to(device)
            xv_q, xm_q = xv_q.to(device), xm_q.to(device)

            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                r_p = model(xv_p, xm_p)
                r_q = model(xv_q, xm_q)

                loss = ulsif_loss(r_p, r_q, RATIO_CLIP) + 0.5 * ULSIF_LAMBDA * l2_penalty(model)

                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            loss_sum += float(loss.item())
            n_steps += 1

        return loss_sum / max(1, n_steps)

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr_loss = run_epoch_ulsif(dl_p_tr, dl_q_tr, opt)
        va_loss = run_epoch_ulsif(dl_p_va, dl_q_va, None) if (len(p_va_idx) > 0 and len(q_va_idx) > 0) else np.nan

        # metrics on all val points using bounded score r/(1+r)
        val_r = predict_ratio_on_indices(model, X_values, X_masks, y_cv, val_idx)
        val_score01 = ratio_to_bounded_score(val_r)
        va_mets = compute_boyce_and_auc(y_cv[val_idx], val_score01)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train uLSIF {tr_loss:.4f} | "
            f"val uLSIF {va_loss:.4f} | "
            f"val AUC {va_mets['ROC_AUC']:.4f}, Boyce {va_mets['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va_mets["ROC_AUC"]) and (va_mets["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": int(in_value_channels),
                "patch_size": int(PATCH_SIZE),
                "emb_dim": int(EMB_DIM),
                "hidden_dims": list(HIDDEN_DIMS),
                "ulsif_lambda": float(ULSIF_LAMBDA),
                "ratio_clip": float(RATIO_CLIP) if RATIO_CLIP is not None else None,
                "clip_g": float(CLIP_G) if CLIP_G is not None else None,
            }
            candidates.append(
                {
                    "loss": float(va_loss) if np.isfinite(va_loss) else float(np.inf),
                    "boyce": float(va_mets["Boyce"]) if np.isfinite(va_mets["Boyce"]) else np.nan,
                    "auc": float(va_mets["ROC_AUC"]) if np.isfinite(va_mets["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_r": val_r,
                    "val_y": y_cv[val_idx].copy(),
                    "epoch": ep,
                }
            )

            # early stopping on uLSIF val loss if available; else on train loss
            key_loss = va_loss if np.isfinite(va_loss) else tr_loss
            if key_loss < best_loss_seen - 1e-9:
                best_loss_seen = float(key_loss)
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"val_uLSIF={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded score for this fold's val subset
    val_r = best["val_r"]
    val_score01 = ratio_to_bounded_score(val_r)
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "ulsif_val_loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score + logratio)
# =========================================================
def ensemble_predict_on_indices_ulsif(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != int(in_value_channels):
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != int(PATCH_SIZE):
            raise RuntimeError(f"Patch size mismatch for fold {fid}")

        model = CNNUlSIF(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", EMB_DIM)),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_g=float(state.get("clip_g", CLIP_G)) if state.get("clip_g", None) is not None else None,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)
    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            ratios = []
            for model in models:
                r = model(xv, xm)
                r = torch.clamp(r, min=1e-12, max=RATIO_CLIP if RATIO_CLIP is not None else 1e18)
                ratios.append(r)

            if ENSEMBLE_IN_LOGSPACE:
                log_r = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_r += weights_t[k] * torch.log(ratios[k])
                r_ens = torch.exp(log_r)
            else:
                r_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    r_ens += weights_t[k] * ratios[k]

            r_ens = torch.clamp(r_ens, min=1e-12, max=RATIO_CLIP if RATIO_CLIP is not None else 1e18)
            logratio = torch.log(r_ens)
            logratio = torch.clamp(logratio, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = r_ens / (1.0 + r_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(logratio.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores for comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_ulsif(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["ulsif_val_loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score r/(1+r)):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen uLSIF val losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_ulsif(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score r/(1+r)):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


# Standard convolution


# Patch size:3


In [ ]:
import os
from typing import List, Dict, Tuple, Optional
import itertools
import warnings

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_3")
TEST_DIR = str(DATA_ROOT / "test_patches_3")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "ulsif" / "cnn_ulsif_patch_models_3")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 3  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Model selection gate (optional)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80

# uLSIF numeric stability
MAX_W = 1e3
EPS_W = 1e-12

# Enforce E_q[w] = 1 (penalty)
USE_EQ1_PENALTY = True
EQ1_LAMBDA = 1.0

# Inference-time normalization:
# - For OOF on CV: uses stored train-q indices (valid on CV arrays)
# - For external TEST: pass Xq_values/Xq_masks explicitly (recommended)
ULSIF_NORMALIZE_EQ1 = True

P_LABEL = 1
Q_LABEL = 0


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers
# =========================================================
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


def iter_cycled(dl_a, dl_b):
    na, nb = len(dl_a), len(dl_b)
    if na == 0 or nb == 0:
        return
    if na >= nb:
        b_it = itertools.cycle(dl_b)
        for a in dl_a:
            yield a, next(b_it)
    else:
        a_it = itertools.cycle(dl_a)
        for b in dl_b:
            yield next(a_it), b


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class PatchXDataset(Dataset):
    def __init__(self, X_values, X_masks, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm


class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNuLSIF(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, max_w=1e3):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.head = MLP(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)
        self.softplus = nn.Softplus(beta=1.0)
        self.max_w = max_w

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        raw = self.head(z)
        w = self.softplus(raw)
        if self.max_w is not None:
            w = torch.clamp(w, 0.0, self.max_w)
        return w


# =========================================================
# uLSIF loss
# =========================================================
def ulsif_loss(
    w_p: torch.Tensor,
    w_q: torch.Tensor,
    use_eq1_penalty: bool = True,
    eq1_lambda: float = 1.0,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    base = 0.5 * (w_q ** 2).mean() - w_p.mean()
    eq1 = (w_q.mean() - 1.0)
    penalty = (eq1 ** 2) * eq1_lambda if use_eq1_penalty else torch.zeros((), device=w_q.device)

    loss = base + penalty
    stats = {
        "base": float(base.detach().cpu()),
        "penalty": float(penalty.detach().cpu()),
        "eq1_err": float(eq1.detach().cpu()),
        "mean_w_p": float(w_p.detach().mean().cpu()),
        "mean_w_q": float(w_q.detach().mean().cpu()),
        "sat_p": float((w_p.detach() >= (MAX_W - 1e-6)).float().mean().cpu()),
        "sat_q": float((w_q.detach() >= (MAX_W - 1e-6)).float().mean().cpu()),
    }
    return loss, stats


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_ulsif_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    p_tr = train_idx[y_cv[train_idx] == P_LABEL]
    q_tr = train_idx[y_cv[train_idx] == Q_LABEL]
    p_va = val_idx[y_cv[val_idx] == P_LABEL]
    q_va = val_idx[y_cv[val_idx] == Q_LABEL]

    if len(p_tr) < 10 or len(q_tr) < 10:
        raise RuntimeError(f"[fold {fold_id}] too few samples: p_tr={len(p_tr)} q_tr={len(q_tr)}")

    print(f"\n[fold {fold_id}] train: p={len(p_tr)} q={len(q_tr)} | val: p={len(p_va)} q={len(q_va)}")

    ds_p_tr = PatchXDataset(X_values, X_masks, p_tr, train=True)
    ds_q_tr = PatchXDataset(X_values, X_masks, q_tr, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_p_tr = DataLoader(ds_p_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_q_tr = DataLoader(ds_q_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNuLSIF(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        max_w=MAX_W,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_train_epoch():
        model.train()
        total_loss = 0.0
        total_steps = 0
        mean_wq_avg = 0.0
        mean_wp_avg = 0.0
        eq1_avg = 0.0

        for (xvp, xmp), (xvq, xmq) in iter_cycled(dl_p_tr, dl_q_tr):
            xvp, xmp = xvp.to(device), xmp.to(device)
            xvq, xmq = xvq.to(device), xmq.to(device)

            opt.zero_grad(set_to_none=True)
            w_p = model(xvp, xmp)
            w_q = model(xvq, xmq)

            loss, st = ulsif_loss(w_p, w_q, USE_EQ1_PENALTY, EQ1_LAMBDA)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += float(loss.detach().cpu())
            mean_wq_avg += st["mean_w_q"]
            mean_wp_avg += st["mean_w_p"]
            eq1_avg += st["eq1_err"]
            total_steps += 1

        denom = max(1, total_steps)
        return {
            "ulsif_loss": total_loss / denom,
            "mean_w_p": mean_wp_avg / denom,
            "mean_w_q": mean_wq_avg / denom,
            "eq1_err": eq1_avg / denom,
        }

    def run_val_metrics():
        model.eval()
        all_w, all_y = [], []
        with torch.no_grad():
            for xv, xm, yb in dl_val:
                xv, xm = xv.to(device), xm.to(device)
                w = model(xv, xm)
                all_w.append(w.detach().cpu().numpy())
                all_y.append(yb.numpy())

        w_np = np.concatenate(all_w)
        y_np = np.concatenate(all_y).astype(int)
        score01 = w_np / (1.0 + w_np)
        mets = compute_boyce_and_auc(y_np, score01)
        return mets, score01

    candidates = []
    best_boyce_key = -np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr = run_train_epoch()
        va, val_score01 = run_val_metrics()

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['ulsif_loss']:.4f}, eq1_err {tr['eq1_err']:+.4f}, "
            f"mean_w_p {tr['mean_w_p']:.4f}, mean_w_q {tr['mean_w_q']:.4f} | "
            f"val AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "max_w": float(MAX_W),
                "p_label": int(P_LABEL),
                "q_label": int(Q_LABEL),
                "method": "uLSIF",
                # IMPORTANT: indices are CV-array indices
                "q_calib_indices": q_tr.astype(np.int64),
            }
            candidates.append(
                {"epoch": ep, "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                 "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                 "state": state, "val_score01": val_score01}
            )

            b = candidates[-1]["boyce"]
            b_key = -np.inf if not np.isfinite(b) else float(b)
            if b_key > best_boyce_key + 1e-9:
                best_boyce_key = b_key
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    def key(c):
        b = c["boyce"]
        a = c["auc"]
        b_val = -np.inf if not np.isfinite(b) else float(b)
        a_val = -np.inf if not np.isfinite(a) else float(a)
        return (b_val, a_val)

    best = max(candidates, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | saved -> {ckpt_path}"
    )

    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": np.nan}
    return best_val, best["val_score01"], val_idx


# =========================================================
# Ensemble prediction
# =========================================================
@torch.no_grad()
def _mean_w_on_samples(model: nn.Module, Xv: np.ndarray, Xm: np.ndarray, batch_size: int) -> float:
    n = Xv.shape[0]
    if n == 0:
        return 1.0
    acc = []
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        xv = torch.from_numpy(Xv[start:end].astype(np.float32)).to(device)
        xm = torch.from_numpy(Xm[start:end].astype(np.float32)).to(device)
        w = model(xv, xm)
        acc.append(w.mean().clamp_min(EPS_W).cpu().item())
    return float(np.mean(acc)) if acc else 1.0


@torch.no_grad()
def _mean_w_on_indices(model: nn.Module, X_values: np.ndarray, X_masks: np.ndarray, idx: np.ndarray, batch_size: int) -> float:
    if idx is None or len(idx) == 0:
        return 1.0
    idx = np.asarray(idx, dtype=np.int64)
    acc = []
    for start in range(0, len(idx), batch_size):
        end = min(start + batch_size, len(idx))
        b = idx[start:end]
        xv = torch.from_numpy(X_values[b].astype(np.float32)).to(device)
        xm = torch.from_numpy(X_masks[b].astype(np.float32)).to(device)
        w = model(xv, xm)
        acc.append(w.mean().clamp_min(EPS_W).cpu().item())
    return float(np.mean(acc)) if acc else 1.0


def ensemble_predict_on_indices_ulsif_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
    # NEW: pass q samples for the CURRENT dataset (recommended for external test)
    Xq_values: Optional[np.ndarray] = None,
    Xq_masks: Optional[np.ndarray] = None,
):
    in_value_channels = X_values.shape[1]

    models = []
    norm_consts = []

    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu", weights_only=False)  # pytorch 2.6+

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")

        model = CNNuLSIF(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            max_w=state.get("max_w", MAX_W),
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        if ULSIF_NORMALIZE_EQ1:
            # Preferred: normalize using q samples from CURRENT dataset
            if Xq_values is not None and Xq_masks is not None:
                c = _mean_w_on_samples(model, Xq_values, Xq_masks, BATCH_SIZE)
            else:
                # Fallback: use stored indices only if they are valid for CURRENT arrays
                q_idx = state.get("q_calib_indices", None)
                if q_idx is not None and len(q_idx) > 0 and int(np.max(q_idx)) < X_values.shape[0]:
                    c = _mean_w_on_indices(model, X_values, X_masks, q_idx, BATCH_SIZE)
                else:
                    warnings.warn(
                        f"[fold {fid}] Skipping Eq1 normalization: checkpoint q_calib_indices "
                        f"do not match current X_values size ({X_values.shape[0]}). "
                        f"Pass Xq_values/Xq_masks for proper normalization."
                    )
                    c = 1.0
        else:
            c = 1.0

        norm_consts.append(float(max(EPS_W, c)))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()
    weights_t = torch.tensor(w, dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)
    all_score01, all_w, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            w_ens = torch.zeros(xv.size(0), device=device)
            for k, model in enumerate(models):
                wk = model(xv, xm) / norm_consts[k]
                w_ens += weights_t[k] * wk

            w_ens = torch.clamp(w_ens, 0.0, MAX_W)
            score01 = w_ens / (1.0 + w_ens)

            all_score01.append(score01.cpu().numpy())
            all_w.append(w_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return np.concatenate(all_score01), np.concatenate(all_w), np.concatenate(all_y)


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)
    fold_aucs, fold_boyces = [], []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_ulsif_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_boyces.append(best_val["Boyce"])
        oof_score01[val_idx] = val_score01

    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score = w/(1+w)):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold best Boyce:", fold_boyces)

    # ----------- Ensemble weights -----------
    weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
    print("\n[Ensemble] Using equal weights:", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_weights_equal.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_weights_equal.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    # IMPORTANT: for TEST normalization, use TEST q-samples (not CV indices)
    Xq_vals_test = X_test[y_test == Q_LABEL]
    Xq_masks_test = M_test[y_test == Q_LABEL]

    test_score01, test_w, test_y = ensemble_predict_on_indices_ulsif_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
        Xq_values=Xq_vals_test,
        Xq_masks=Xq_masks_test,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score = w/(1+w)):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_w_ens.npy"), test_w)
    print("[Test] Saved test_score01.npy, test_w_ens.npy")


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING (uLSIF CNN)
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "ulsif" / "cnn_ulsif_patch_models_3_nn")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "ulsif" / "dengue_cnn_ulsif_suitability_3_nn.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
PATCH_SIZE     = 3
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training
EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

# uLSIF numeric stability (must match training / checkpoint intent)
MAX_W = 1e3
EPS_W = 1e-12

# Output control:
#   - If WRITE_W=True: write w(x) (ratio), clamped to [0, MAX_W]
#   - Else: write bounded score = w/(1+w) in (0,1)
WRITE_W = False

# Optional: normalize so E_q[w]=1 in the raster output.
# This requires you to define what "q" means in raster context.
# Common choice: land pixels (or land pixels with some mask).
ULSIF_NORMALIZE_EQ1 = False

# If normalization is enabled, choose q pixels via landmask==1 (default here).
# If you want a different q-region, change this selector.
NORMALIZE_Q_IS_LAND = True


# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)


# -------------------------------------------------------------
# Model definitions — MUST MATCH TRAINING
# -------------------------------------------------------------
class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6 (comment from older version; actual depends on patch size)

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNuLSIF(nn.Module):
    """
    Neural uLSIF: network outputs w(x) >= 0 directly.
    """
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, max_w=1e3):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.head = MLP(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)
        self.softplus = nn.Softplus(beta=1.0)
        self.max_w = max_w

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        raw = self.head(z)
        w = self.softplus(raw)  # >= 0
        if self.max_w is not None:
            w = torch.clamp(w, 0.0, self.max_w)
        return w


def w_to_score01(w: torch.Tensor) -> torch.Tensor:
    # bounded monotone map used in training metrics: w/(1+w)
    w = torch.clamp(w, 0.0, MAX_W)
    return w / (1.0 + w)


# -------------------------------------------------------------
# Load ensemble models (uLSIF)
# Expects:
#   - cnn_ulsif_fold_<fid>.pt
#   - fold_ids.npy
#   - fold_weights_equal.npy  (or change below)
# -------------------------------------------------------------
def load_ulsif_ensemble_models(model_dir: str) -> Tuple[List[CNNuLSIF], torch.Tensor, int, int]:
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    # training script saved "fold_weights_equal.npy"
    w_path = mdir / "fold_weights_equal.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path} (expected from your training script)")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if (not np.isfinite(weights).any()) or float(np.nansum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        w = np.maximum(weights, 1e-8)
        weights = w / w.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded weights:", weights.tolist())

    # Read an example ckpt for architecture params
    example_ckpt = torch.load(mdir / f"cnn_ulsif_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example_ckpt["in_value_channels"])
    patch_size = int(example_ckpt.get("patch_size", PATCH_SIZE))
    emb_dim = int(example_ckpt.get("emb_dim", EMB_DIM))
    hidden_dims = tuple(example_ckpt.get("hidden_dims", HIDDEN_DIMS))
    max_w = float(example_ckpt.get("max_w", MAX_W))

    models: List[CNNuLSIF] = []
    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_ulsif_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")

        model = CNNuLSIF(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,  # eval() disables dropout anyway
            max_w=state.get("max_w", max_w),
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    return models, weights_t, in_value_channels, patch_size


# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)


# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks


# -------------------------------------------------------------
# Estimate normalization constant so mean_q[w]=1 (optional)
# Here we take q as land pixels (landmask==1) by default.
# This is NOT "part of uLSIF", but can stabilize map scale.
# -------------------------------------------------------------
def estimate_eq1_norm_const(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    landmask: np.ndarray,
    models: List[CNNuLSIF],
    weights_t: torch.Tensor,
    patch_size: int,
    tile_size: int,
    batch_size: int,
) -> float:
    if not NORMALIZE_Q_IS_LAND:
        raise NotImplementedError("Only NORMALIZE_Q_IS_LAND=True implemented in this snippet.")

    template = rasters[0]
    H, W = template.height, template.width

    total_sum = 0.0
    total_n = 0

    n_rows_tiles = math.ceil(H / tile_size)
    n_cols_tiles = math.ceil(W / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Norm tiles (E_q[w])", unit="tile") as pbar:
        for row0 in range(0, H, tile_size):
            row1 = min(row0 + tile_size, H)
            tile_h = row1 - row0

            for col0 in range(0, W, tile_size):
                col1 = min(col0 + tile_size, W)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_q = (lm_tile == 1)
                if not np.any(is_q):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_q)
                N_q = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                with torch.no_grad():
                    for start in range(0, N_q, batch_size):
                        end = min(start + batch_size, N_q)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        w_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                        for k, model in enumerate(models):
                            w_ens += weights_t[k] * model(xv, xm)

                        w_ens = torch.clamp(w_ens, 0.0, MAX_W)

                        total_sum += float(w_ens.sum().cpu())
                        total_n += int(w_ens.numel())

                pbar.update(1)

    if total_n == 0:
        return 1.0
    mean_w = total_sum / total_n
    return float(max(mean_w, EPS_W))


# -------------------------------------------------------------
# Main prediction over rasters (uLSIF)
# -------------------------------------------------------------
def predict_suitability_map_ulsif_cnn(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load ensemble (uLSIF)
    models, weights_t, in_value_channels, patch_size = load_ulsif_ensemble_models(model_dir=model_dir)
    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )
    if patch_size != PATCH_SIZE:
        raise RuntimeError(f"Patch size from checkpoint ({patch_size}) != PATCH_SIZE ({PATCH_SIZE}).")

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Optional normalization constant so mean_q[w]=1
    norm_const = 1.0
    if ULSIF_NORMALIZE_EQ1:
        norm_const = estimate_eq1_norm_const(
            rasters=rasters,
            raster_medians=raster_medians,
            landmask=landmask,
            models=models,
            weights_t=weights_t,
            patch_size=patch_size,
            tile_size=tile_size,
            batch_size=batch_size,
        )
        print(f"[Normalize] Estimated mean_q[w] = {norm_const:.6g} -> using w /= {norm_const:.6g}")

    # Map of raw w (ratio)
    w_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_wvals = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        w_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                        for k, model in enumerate(models):
                            w_ens += weights_t[k] * model(xv, xm)

                        w_ens = w_ens / float(norm_const)
                        w_ens = torch.clamp(w_ens, 0.0, MAX_W)
                        tile_wvals[start:end] = w_ens.detach().cpu().numpy().astype("float32")

                tile_w_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_w_arr[rr_rel, cc_rel] = tile_wvals
                w_map[row0:row1, col0:col1] = tile_w_arr

                pbar.update(1)

    # ---------------------------------------------------------
    # Output selection
    # ---------------------------------------------------------
    if WRITE_W:
        out_map = w_map.astype(np.float32)
        out_name = "uLSIF ratio w(x)"
    else:
        # bounded suitability = w/(1+w) in (0,1)
        x = torch.from_numpy(w_map.astype(np.float32))
        out_map = (x / (1.0 + x)).cpu().numpy().astype(np.float32)
        out_name = "bounded score = w/(1+w) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()


if __name__ == "__main__":
    predict_suitability_map_ulsif_cnn(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING (uLSIF CNN)
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "ulsif" / "cnn_ulsif_patch_models_3_nn")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "ulsif" / "dengue_cnn_ulsif_suitability_3_nn.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
PATCH_SIZE     = 3
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training
EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

# uLSIF numeric stability
MAX_W = 1e3
EPS_W = 1e-12

# Output:
#   - If WRITE_W=True: write w(x)
#   - Else: write bounded score = w/(1+w)
WRITE_W = False

# Optional: normalize so E_q[w]=1 in the raster output.
# Here q is defined in raster context (default: land pixels).
ULSIF_NORMALIZE_EQ1 = False
NORMALIZE_Q_IS_LAND = True

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)


# -------------------------------------------------------------
# Model definitions — MUST MATCH TRAINING
# -------------------------------------------------------------
class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNuLSIF(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, max_w=1e3):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.head = MLP(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)
        self.softplus = nn.Softplus(beta=1.0)
        self.max_w = max_w

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        raw = self.head(z)
        w = self.softplus(raw)
        if self.max_w is not None:
            w = torch.clamp(w, 0.0, self.max_w)
        return w


def w_to_score01(w: torch.Tensor) -> torch.Tensor:
    w = torch.clamp(w, 0.0, MAX_W)
    return w / (1.0 + w)


# -------------------------------------------------------------
# Load ensemble models (uLSIF)
# PyTorch 2.6+: torch.load(..., weights_only=False)
# -------------------------------------------------------------
def load_ulsif_ensemble_models(model_dir: str) -> Tuple[List[CNNuLSIF], torch.Tensor, int, int, np.ndarray]:
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_equal.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if (not np.isfinite(weights).any()) or float(np.nansum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        w = np.maximum(weights, 1e-8)
        weights = w / w.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded weights:", weights.tolist())

    # NOTE: weights_only=False because checkpoint contains numpy arrays / metadata
    example_ckpt = torch.load(mdir / f"cnn_ulsif_fold_{fold_ids[0]}.pt", map_location="cpu", weights_only=False)
    in_value_channels = int(example_ckpt["in_value_channels"])
    patch_size = int(example_ckpt.get("patch_size", PATCH_SIZE))
    emb_dim = int(example_ckpt.get("emb_dim", EMB_DIM))
    hidden_dims = tuple(example_ckpt.get("hidden_dims", HIDDEN_DIMS))
    max_w = float(example_ckpt.get("max_w", MAX_W))

    models: List[CNNuLSIF] = []
    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_ulsif_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu", weights_only=False)

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")

        model = CNNuLSIF(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,
            max_w=state.get("max_w", max_w),
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    return models, weights_t, in_value_channels, patch_size, fold_ids


# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)


# -------------------------------------------------------------
# Extract patches for a tile's selected pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N = rr_rel.shape[0]
    values = np.zeros((N, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks


# -------------------------------------------------------------
# Per-fold Eq1 normalization constant on raster-defined q
# q defaults to land pixels (landmask==1)
# This fixes the "CV indices don't apply to raster dataset" issue.
# -------------------------------------------------------------
@torch.no_grad()
def estimate_eq1_norm_const_per_fold(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    landmask: np.ndarray,
    models: List[CNNuLSIF],
    patch_size: int,
    tile_size: int,
    batch_size: int,
) -> np.ndarray:
    if not NORMALIZE_Q_IS_LAND:
        raise NotImplementedError("Only NORMALIZE_Q_IS_LAND=True implemented.")

    template = rasters[0]
    H, W = template.height, template.width

    # accumulate per model
    total_sum = np.zeros(len(models), dtype=np.float64)
    total_n = 0

    n_rows_tiles = math.ceil(H / tile_size)
    n_cols_tiles = math.ceil(W / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Norm tiles (per-fold E_q[w])", unit="tile") as pbar:
        for row0 in range(0, H, tile_size):
            row1 = min(row0 + tile_size, H)

            for col0 in range(0, W, tile_size):
                col1 = min(col0 + tile_size, W)

                lm_tile = landmask[row0:row1, col0:col1]
                is_q = (lm_tile == 1)
                if not np.any(is_q):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_q)
                N_q = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                for start in range(0, N_q, batch_size):
                    end = min(start + batch_size, N_q)
                    xv = torch.from_numpy(vals_tile[start:end]).to(device)
                    xm = torch.from_numpy(masks_tile[start:end]).to(device)

                    for k, model in enumerate(models):
                        wk = model(xv, xm)  # already clamped in model
                        wk = torch.clamp(wk, 0.0, MAX_W)
                        total_sum[k] += float(wk.sum().cpu())

                    total_n += int(xv.size(0))

                pbar.update(1)

    if total_n == 0:
        return np.ones(len(models), dtype=np.float64)

    mean_w = total_sum / float(total_n)
    mean_w = np.maximum(mean_w, EPS_W)
    return mean_w.astype(np.float64)


# -------------------------------------------------------------
# Main prediction over rasters (uLSIF)
# -------------------------------------------------------------
def predict_suitability_map_ulsif_cnn(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load ensemble (uLSIF)
    models, weights_t, in_value_channels, patch_size, fold_ids = load_ulsif_ensemble_models(model_dir=model_dir)
    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )
    if patch_size != PATCH_SIZE:
        raise RuntimeError(f"Patch size from checkpoint ({patch_size}) != PATCH_SIZE ({PATCH_SIZE}).")

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Per-fold Eq1 normalization constants on raster-defined q
    # This is the correct analogue of "pass Xq_values/Xq_masks" in raster setting.
    per_fold_norm = np.ones(len(models), dtype=np.float64)
    if ULSIF_NORMALIZE_EQ1:
        per_fold_norm = estimate_eq1_norm_const_per_fold(
            rasters=rasters,
            raster_medians=raster_medians,
            landmask=landmask,
            models=models,
            patch_size=patch_size,
            tile_size=tile_size,
            batch_size=batch_size,
        )
        print("[Normalize] Estimated per-fold mean_q[w] (land as q):")
        for fid, c in zip(fold_ids.tolist(), per_fold_norm.tolist()):
            print(f"  fold {fid}: {c:.6g}  (will use wk /= {c:.6g})")
        per_fold_norm = np.maximum(per_fold_norm, EPS_W)

    per_fold_norm_t = torch.tensor(per_fold_norm, dtype=torch.float32, device=device)

    # Map of raw w (ratio)
    w_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_wvals = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # ensemble of (optionally normalized) fold outputs
                        w_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                        for k, model in enumerate(models):
                            wk = model(xv, xm)
                            wk = torch.clamp(wk, 0.0, MAX_W)
                            if ULSIF_NORMALIZE_EQ1:
                                wk = wk / per_fold_norm_t[k]
                            w_ens += weights_t[k] * wk

                        w_ens = torch.clamp(w_ens, 0.0, MAX_W)
                        tile_wvals[start:end] = w_ens.detach().cpu().numpy().astype("float32")

                tile_w_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_w_arr[rr_rel, cc_rel] = tile_wvals
                w_map[row0:row1, col0:col1] = tile_w_arr

                pbar.update(1)

    # ---------------------------------------------------------
    # Output selection
    # ---------------------------------------------------------
    if WRITE_W:
        out_map = w_map.astype(np.float32)
        out_name = "uLSIF ratio w(x)"
    else:
        x = torch.from_numpy(w_map.astype(np.float32))
        out_map = (x / (1.0 + x)).cpu().numpy().astype(np.float32)
        out_name = "bounded score = w/(1+w) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()


if __name__ == "__main__":
    predict_suitability_map_ulsif_cnn(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


# Patch size: 5


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_23")
TEST_DIR = str(DATA_ROOT / "test_patches_23")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "ulsif" / "cnn_ulsif_patch_models_23")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 23  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Model selection gate (optional): computed on bounded monotone score = w/(1+w)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80

# uLSIF numeric stability
MAX_W = 1e3      # clamp w(x) to avoid blow-ups (tune if needed)
EPS_W = 1e-12

# Optional: normalize so E_q[w]=1 at inference time (NOT part of vanilla uLSIF)
ULSIF_NORMALIZE_EQ1 = False

# If your labels are reversed, flip these:
#   y==1 -> p (target)
#   y==0 -> q (reference)
P_LABEL = 1
Q_LABEL = 0


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers
# =========================================================
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    """Returns xv, xm, y. Useful for validation metrics only."""
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class PatchXDataset(Dataset):
    """Returns xv, xm only. Used for uLSIF p/q loaders."""
    def __init__(self, X_values, X_masks, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm


class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNuLSIF(nn.Module):
    """
    Neural uLSIF: network outputs w(x) >= 0 directly.
    Uses softplus for nonnegativity, plus clamp for stability.
    """
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, max_w=1e3):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.head = MLP(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)
        self.softplus = nn.Softplus(beta=1.0)
        self.max_w = max_w

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        raw = self.head(z)
        w = self.softplus(raw)  # >= 0
        if self.max_w is not None:
            w = torch.clamp(w, 0.0, self.max_w)
        return w


# =========================================================
# uLSIF loss
#   minimize 0.5 E_q[w^2] - E_p[w]
# =========================================================
def ulsif_loss(w_p: torch.Tensor, w_q: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, float]]:
    loss = 0.5 * (w_q ** 2).mean() - w_p.mean()
    stats = {
        "mean_w_p": float(w_p.detach().mean().cpu()),
        "mean_w_q": float(w_q.detach().mean().cpu()),
        "mean_wq2": float((w_q.detach() ** 2).mean().cpu()),
    }
    return loss, stats


# =========================================================
# Training per fold (uLSIF)
# - train on p/q batches
# - validation metrics computed on bounded score = w/(1+w)
# - returns OOF bounded score for that fold val split
# =========================================================
def train_and_save_fold_model_ulsif_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # p/q split inside train
    p_tr = train_idx[y_cv[train_idx] == P_LABEL]
    q_tr = train_idx[y_cv[train_idx] == Q_LABEL]
    p_va = val_idx[y_cv[val_idx] == P_LABEL]
    q_va = val_idx[y_cv[val_idx] == Q_LABEL]

    if len(p_tr) < 10 or len(q_tr) < 10:
        raise RuntimeError(f"[fold {fold_id}] too few samples: p_tr={len(p_tr)} q_tr={len(q_tr)}")

    print(f"\n[fold {fold_id}] train: p={len(p_tr)} q={len(q_tr)} | val: p={len(p_va)} q={len(q_va)}")

    ds_p_tr = PatchXDataset(X_values, X_masks, p_tr, train=True)
    ds_q_tr = PatchXDataset(X_values, X_masks, q_tr, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_p_tr = DataLoader(ds_p_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_q_tr = DataLoader(ds_q_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNuLSIF(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        max_w=MAX_W,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_train_epoch():
        model.train()
        total_loss = 0.0
        total_steps = 0
        mean_wq_avg = 0.0
        mean_wp_avg = 0.0

        for (xvp, xmp), (xvq, xmq) in zip(dl_p_tr, dl_q_tr):
            xvp, xmp = xvp.to(device), xmp.to(device)
            xvq, xmq = xvq.to(device), xmq.to(device)

            opt.zero_grad(set_to_none=True)
            w_p = model(xvp, xmp)
            w_q = model(xvq, xmq)
            loss, st = ulsif_loss(w_p, w_q)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += float(loss.detach().cpu())
            mean_wq_avg += st["mean_w_q"]
            mean_wp_avg += st["mean_w_p"]
            total_steps += 1

        return {
            "ulsif_loss": total_loss / max(1, total_steps),
            "mean_w_p": mean_wp_avg / max(1, total_steps),
            "mean_w_q": mean_wq_avg / max(1, total_steps),
        }

    def run_val_metrics():
        model.eval()
        all_w, all_y = [], []
        with torch.no_grad():
            for xv, xm, yb in dl_val:
                xv, xm = xv.to(device), xm.to(device)
                w = model(xv, xm)
                all_w.append(w.detach().cpu().numpy())
                all_y.append(yb.numpy())

        w_np = np.concatenate(all_w)
        y_np = np.concatenate(all_y).astype(int)

        # bounded monotone score in (0,1) for Boyce/AUC comparability
        score01 = w_np / (1.0 + w_np)
        mets = compute_boyce_and_auc(y_np, score01)
        return mets, score01, w_np, y_np

    candidates = []
    best_boyce_key = -np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr = run_train_epoch()
        va, val_score01, val_w, val_y = run_val_metrics()

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['ulsif_loss']:.4f}, mean_w_p {tr['mean_w_p']:.4f}, mean_w_q {tr['mean_w_q']:.4f} | "
            f"val AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "max_w": float(MAX_W),
                "p_label": int(P_LABEL),
                "q_label": int(Q_LABEL),
                "method": "uLSIF",
            }
            candidates.append(
                {
                    "epoch": ep,
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_score01": val_score01,
                    "val_w": val_w,
                    "val_y": val_y,
                }
            )

            b = candidates[-1]["boyce"]
            b_key = -np.inf if not np.isfinite(b) else float(b)
            if b_key > best_boyce_key + 1e-9:
                best_boyce_key = b_key
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    # select by Boyce, then AUC
    def key(c):
        b = c["boyce"]
        a = c["auc"]
        b_val = -np.inf if not np.isfinite(b) else float(b)
        a_val = -np.inf if not np.isfinite(a) else float(a)
        return (b_val, a_val)

    best = max(candidates, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | saved -> {ckpt_path}"
    )

    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": np.nan}
    return best_val, best["val_score01"], val_idx


# =========================================================
# Ensemble prediction (uLSIF)
# - each model outputs w(x) >= 0
# - ensemble = weighted mean of w
# - bounded score = w/(1+w)
# - optional normalization: divide by mean_q(w) so E_q[w]=1
# =========================================================
def ensemble_predict_on_indices_ulsif_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
    # If ULSIF_NORMALIZE_EQ1=True, pass q samples here (arrays, not indices):
    Xq_values: np.ndarray = None,
    Xq_masks: np.ndarray = None,
):
    in_value_channels = X_values.shape[1]

    models = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")

        model = CNNuLSIF(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            max_w=state.get("max_w", MAX_W),
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()
    weights_t = torch.tensor(w, dtype=torch.float32, device=device)

    # optional normalization constant
    norm_const = 1.0
    if ULSIF_NORMALIZE_EQ1:
        if Xq_values is None or Xq_masks is None:
            raise ValueError("ULSIF_NORMALIZE_EQ1=True requires Xq_values and Xq_masks (q-samples).")
        with torch.no_grad():
            xv_q = torch.from_numpy(Xq_values.astype(np.float32)).to(device)
            xm_q = torch.from_numpy(Xq_masks.astype(np.float32)).to(device)

            wq_ens = torch.zeros(xv_q.size(0), device=device)
            for k, model in enumerate(models):
                wq_ens += weights_t[k] * model(xv_q, xm_q)

            norm_const = float(wq_ens.mean().clamp_min(EPS_W).cpu())

    idx = np.asarray(indices, dtype=np.int64)
    all_score01, all_w, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            w_ens = torch.zeros(xv.size(0), device=device)
            for k, model in enumerate(models):
                w_ens += weights_t[k] * model(xv, xm)

            w_ens = w_ens / norm_const
            w_ens = torch.clamp(w_ens, 0.0, MAX_W)

            score01 = w_ens / (1.0 + w_ens)

            all_score01.append(score01.cpu().numpy())
            all_w.append(w_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return np.concatenate(all_score01), np.concatenate(all_w), np.concatenate(all_y)


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for metrics comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_boyces = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_ulsif_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_boyces.append(best_val["Boyce"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score = w/(1+w)):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold best Boyce:", fold_boyces)

    # ----------- Ensemble weights -----------
    # uLSIF loss is comparable across folds in principle, but to match your KLIEP setup we keep equal weights.
    weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
    print("\n[Ensemble] Using equal weights:", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_weights_equal.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_weights_equal.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    # If you want E_q[w]=1 normalization on test set, uncomment:
    # ULSIF_NORMALIZE_EQ1 = True
    # Xq_vals = X_test[y_test == Q_LABEL]
    # Xq_masks = M_test[y_test == Q_LABEL]

    test_score01, test_w, test_y = ensemble_predict_on_indices_ulsif_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
        # Xq_values=Xq_vals,
        # Xq_masks=Xq_masks,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score = w/(1+w)):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_w_ens.npy"), test_w)
    print("[Test] Saved test_score01.npy, test_w_ens.npy")


# Patch size: 13


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_13")
TEST_DIR = str(DATA_ROOT / "test_patches_13")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "ulsif" / "cnn_ulsif_patch_models_13")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 13 # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Model selection gate (optional): computed on bounded monotone score = w/(1+w)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80

# uLSIF numeric stability
MAX_W = 1e3      # clamp w(x) to avoid blow-ups (tune if needed)
EPS_W = 1e-12

# Optional: normalize so E_q[w]=1 at inference time (NOT part of vanilla uLSIF)
ULSIF_NORMALIZE_EQ1 = False

# If your labels are reversed, flip these:
#   y==1 -> p (target)
#   y==0 -> q (reference)
P_LABEL = 1
Q_LABEL = 0


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers
# =========================================================
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    """Returns xv, xm, y. Useful for validation metrics only."""
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class PatchXDataset(Dataset):
    """Returns xv, xm only. Used for uLSIF p/q loaders."""
    def __init__(self, X_values, X_masks, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm


class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNuLSIF(nn.Module):
    """
    Neural uLSIF: network outputs w(x) >= 0 directly.
    Uses softplus for nonnegativity, plus clamp for stability.
    """
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, max_w=1e3):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.head = MLP(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)
        self.softplus = nn.Softplus(beta=1.0)
        self.max_w = max_w

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        raw = self.head(z)
        w = self.softplus(raw)  # >= 0
        if self.max_w is not None:
            w = torch.clamp(w, 0.0, self.max_w)
        return w


# =========================================================
# uLSIF loss
#   minimize 0.5 E_q[w^2] - E_p[w]
# =========================================================
def ulsif_loss(w_p: torch.Tensor, w_q: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, float]]:
    loss = 0.5 * (w_q ** 2).mean() - w_p.mean()
    stats = {
        "mean_w_p": float(w_p.detach().mean().cpu()),
        "mean_w_q": float(w_q.detach().mean().cpu()),
        "mean_wq2": float((w_q.detach() ** 2).mean().cpu()),
    }
    return loss, stats


# =========================================================
# Training per fold (uLSIF)
# - train on p/q batches
# - validation metrics computed on bounded score = w/(1+w)
# - returns OOF bounded score for that fold val split
# =========================================================
def train_and_save_fold_model_ulsif_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # p/q split inside train
    p_tr = train_idx[y_cv[train_idx] == P_LABEL]
    q_tr = train_idx[y_cv[train_idx] == Q_LABEL]
    p_va = val_idx[y_cv[val_idx] == P_LABEL]
    q_va = val_idx[y_cv[val_idx] == Q_LABEL]

    if len(p_tr) < 10 or len(q_tr) < 10:
        raise RuntimeError(f"[fold {fold_id}] too few samples: p_tr={len(p_tr)} q_tr={len(q_tr)}")

    print(f"\n[fold {fold_id}] train: p={len(p_tr)} q={len(q_tr)} | val: p={len(p_va)} q={len(q_va)}")

    ds_p_tr = PatchXDataset(X_values, X_masks, p_tr, train=True)
    ds_q_tr = PatchXDataset(X_values, X_masks, q_tr, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_p_tr = DataLoader(ds_p_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_q_tr = DataLoader(ds_q_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNuLSIF(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        max_w=MAX_W,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_train_epoch():
        model.train()
        total_loss = 0.0
        total_steps = 0
        mean_wq_avg = 0.0
        mean_wp_avg = 0.0

        for (xvp, xmp), (xvq, xmq) in zip(dl_p_tr, dl_q_tr):
            xvp, xmp = xvp.to(device), xmp.to(device)
            xvq, xmq = xvq.to(device), xmq.to(device)

            opt.zero_grad(set_to_none=True)
            w_p = model(xvp, xmp)
            w_q = model(xvq, xmq)
            loss, st = ulsif_loss(w_p, w_q)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += float(loss.detach().cpu())
            mean_wq_avg += st["mean_w_q"]
            mean_wp_avg += st["mean_w_p"]
            total_steps += 1

        return {
            "ulsif_loss": total_loss / max(1, total_steps),
            "mean_w_p": mean_wp_avg / max(1, total_steps),
            "mean_w_q": mean_wq_avg / max(1, total_steps),
        }

    def run_val_metrics():
        model.eval()
        all_w, all_y = [], []
        with torch.no_grad():
            for xv, xm, yb in dl_val:
                xv, xm = xv.to(device), xm.to(device)
                w = model(xv, xm)
                all_w.append(w.detach().cpu().numpy())
                all_y.append(yb.numpy())

        w_np = np.concatenate(all_w)
        y_np = np.concatenate(all_y).astype(int)

        # bounded monotone score in (0,1) for Boyce/AUC comparability
        score01 = w_np / (1.0 + w_np)
        mets = compute_boyce_and_auc(y_np, score01)
        return mets, score01, w_np, y_np

    candidates = []
    best_boyce_key = -np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr = run_train_epoch()
        va, val_score01, val_w, val_y = run_val_metrics()

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['ulsif_loss']:.4f}, mean_w_p {tr['mean_w_p']:.4f}, mean_w_q {tr['mean_w_q']:.4f} | "
            f"val AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "max_w": float(MAX_W),
                "p_label": int(P_LABEL),
                "q_label": int(Q_LABEL),
                "method": "uLSIF",
            }
            candidates.append(
                {
                    "epoch": ep,
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_score01": val_score01,
                    "val_w": val_w,
                    "val_y": val_y,
                }
            )

            b = candidates[-1]["boyce"]
            b_key = -np.inf if not np.isfinite(b) else float(b)
            if b_key > best_boyce_key + 1e-9:
                best_boyce_key = b_key
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    # select by Boyce, then AUC
    def key(c):
        b = c["boyce"]
        a = c["auc"]
        b_val = -np.inf if not np.isfinite(b) else float(b)
        a_val = -np.inf if not np.isfinite(a) else float(a)
        return (b_val, a_val)

    best = max(candidates, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | saved -> {ckpt_path}"
    )

    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": np.nan}
    return best_val, best["val_score01"], val_idx


# =========================================================
# Ensemble prediction (uLSIF)
# - each model outputs w(x) >= 0
# - ensemble = weighted mean of w
# - bounded score = w/(1+w)
# - optional normalization: divide by mean_q(w) so E_q[w]=1
# =========================================================
def ensemble_predict_on_indices_ulsif_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
    # If ULSIF_NORMALIZE_EQ1=True, pass q samples here (arrays, not indices):
    Xq_values: np.ndarray = None,
    Xq_masks: np.ndarray = None,
):
    in_value_channels = X_values.shape[1]

    models = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")

        model = CNNuLSIF(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            max_w=state.get("max_w", MAX_W),
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()
    weights_t = torch.tensor(w, dtype=torch.float32, device=device)

    # optional normalization constant
    norm_const = 1.0
    if ULSIF_NORMALIZE_EQ1:
        if Xq_values is None or Xq_masks is None:
            raise ValueError("ULSIF_NORMALIZE_EQ1=True requires Xq_values and Xq_masks (q-samples).")
        with torch.no_grad():
            xv_q = torch.from_numpy(Xq_values.astype(np.float32)).to(device)
            xm_q = torch.from_numpy(Xq_masks.astype(np.float32)).to(device)

            wq_ens = torch.zeros(xv_q.size(0), device=device)
            for k, model in enumerate(models):
                wq_ens += weights_t[k] * model(xv_q, xm_q)

            norm_const = float(wq_ens.mean().clamp_min(EPS_W).cpu())

    idx = np.asarray(indices, dtype=np.int64)
    all_score01, all_w, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            w_ens = torch.zeros(xv.size(0), device=device)
            for k, model in enumerate(models):
                w_ens += weights_t[k] * model(xv, xm)

            w_ens = w_ens / norm_const
            w_ens = torch.clamp(w_ens, 0.0, MAX_W)

            score01 = w_ens / (1.0 + w_ens)

            all_score01.append(score01.cpu().numpy())
            all_w.append(w_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return np.concatenate(all_score01), np.concatenate(all_w), np.concatenate(all_y)


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for metrics comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_boyces = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_ulsif_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_boyces.append(best_val["Boyce"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score = w/(1+w)):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold best Boyce:", fold_boyces)

    # ----------- Ensemble weights -----------
    # uLSIF loss is comparable across folds in principle, but to match your KLIEP setup we keep equal weights.
    weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
    print("\n[Ensemble] Using equal weights:", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_weights_equal.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_weights_equal.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    # If you want E_q[w]=1 normalization on test set, uncomment:
    # ULSIF_NORMALIZE_EQ1 = True
    # Xq_vals = X_test[y_test == Q_LABEL]
    # Xq_masks = M_test[y_test == Q_LABEL]

    test_score01, test_w, test_y = ensemble_predict_on_indices_ulsif_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
        # Xq_values=Xq_vals,
        # Xq_masks=Xq_masks,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score = w/(1+w)):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_w_ens.npy"), test_w)
    print("[Test] Saved test_score01.npy, test_w_ens.npy")


# Patch size 33


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_33")
TEST_DIR = str(DATA_ROOT / "test_patches_33")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "ulsif" / "cnn_ulsif_patch_models_33")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 33  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Model selection gate (optional): computed on bounded monotone score = w/(1+w)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80

# uLSIF numeric stability
MAX_W = 1e3      # clamp w(x) to avoid blow-ups (tune if needed)
EPS_W = 1e-12

# Optional: normalize so E_q[w]=1 at inference time (NOT part of vanilla uLSIF)
ULSIF_NORMALIZE_EQ1 = False

# If your labels are reversed, flip these:
#   y==1 -> p (target)
#   y==0 -> q (reference)
P_LABEL = 1
Q_LABEL = 0


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers
# =========================================================
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    """Returns xv, xm, y. Useful for validation metrics only."""
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class PatchXDataset(Dataset):
    """Returns xv, xm only. Used for uLSIF p/q loaders."""
    def __init__(self, X_values, X_masks, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm


class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNuLSIF(nn.Module):
    """
    Neural uLSIF: network outputs w(x) >= 0 directly.
    Uses softplus for nonnegativity, plus clamp for stability.
    """
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, max_w=1e3):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.head = MLP(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)
        self.softplus = nn.Softplus(beta=1.0)
        self.max_w = max_w

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        raw = self.head(z)
        w = self.softplus(raw)  # >= 0
        if self.max_w is not None:
            w = torch.clamp(w, 0.0, self.max_w)
        return w


# =========================================================
# uLSIF loss
#   minimize 0.5 E_q[w^2] - E_p[w]
# =========================================================
def ulsif_loss(w_p: torch.Tensor, w_q: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, float]]:
    loss = 0.5 * (w_q ** 2).mean() - w_p.mean()
    stats = {
        "mean_w_p": float(w_p.detach().mean().cpu()),
        "mean_w_q": float(w_q.detach().mean().cpu()),
        "mean_wq2": float((w_q.detach() ** 2).mean().cpu()),
    }
    return loss, stats


# =========================================================
# Training per fold (uLSIF)
# - train on p/q batches
# - validation metrics computed on bounded score = w/(1+w)
# - returns OOF bounded score for that fold val split
# =========================================================
def train_and_save_fold_model_ulsif_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # p/q split inside train
    p_tr = train_idx[y_cv[train_idx] == P_LABEL]
    q_tr = train_idx[y_cv[train_idx] == Q_LABEL]
    p_va = val_idx[y_cv[val_idx] == P_LABEL]
    q_va = val_idx[y_cv[val_idx] == Q_LABEL]

    if len(p_tr) < 10 or len(q_tr) < 10:
        raise RuntimeError(f"[fold {fold_id}] too few samples: p_tr={len(p_tr)} q_tr={len(q_tr)}")

    print(f"\n[fold {fold_id}] train: p={len(p_tr)} q={len(q_tr)} | val: p={len(p_va)} q={len(q_va)}")

    ds_p_tr = PatchXDataset(X_values, X_masks, p_tr, train=True)
    ds_q_tr = PatchXDataset(X_values, X_masks, q_tr, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_p_tr = DataLoader(ds_p_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_q_tr = DataLoader(ds_q_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNuLSIF(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        max_w=MAX_W,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_train_epoch():
        model.train()
        total_loss = 0.0
        total_steps = 0
        mean_wq_avg = 0.0
        mean_wp_avg = 0.0

        for (xvp, xmp), (xvq, xmq) in zip(dl_p_tr, dl_q_tr):
            xvp, xmp = xvp.to(device), xmp.to(device)
            xvq, xmq = xvq.to(device), xmq.to(device)

            opt.zero_grad(set_to_none=True)
            w_p = model(xvp, xmp)
            w_q = model(xvq, xmq)
            loss, st = ulsif_loss(w_p, w_q)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += float(loss.detach().cpu())
            mean_wq_avg += st["mean_w_q"]
            mean_wp_avg += st["mean_w_p"]
            total_steps += 1

        return {
            "ulsif_loss": total_loss / max(1, total_steps),
            "mean_w_p": mean_wp_avg / max(1, total_steps),
            "mean_w_q": mean_wq_avg / max(1, total_steps),
        }

    def run_val_metrics():
        model.eval()
        all_w, all_y = [], []
        with torch.no_grad():
            for xv, xm, yb in dl_val:
                xv, xm = xv.to(device), xm.to(device)
                w = model(xv, xm)
                all_w.append(w.detach().cpu().numpy())
                all_y.append(yb.numpy())

        w_np = np.concatenate(all_w)
        y_np = np.concatenate(all_y).astype(int)

        # bounded monotone score in (0,1) for Boyce/AUC comparability
        score01 = w_np / (1.0 + w_np)
        mets = compute_boyce_and_auc(y_np, score01)
        return mets, score01, w_np, y_np

    candidates = []
    best_boyce_key = -np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr = run_train_epoch()
        va, val_score01, val_w, val_y = run_val_metrics()

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['ulsif_loss']:.4f}, mean_w_p {tr['mean_w_p']:.4f}, mean_w_q {tr['mean_w_q']:.4f} | "
            f"val AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "max_w": float(MAX_W),
                "p_label": int(P_LABEL),
                "q_label": int(Q_LABEL),
                "method": "uLSIF",
            }
            candidates.append(
                {
                    "epoch": ep,
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_score01": val_score01,
                    "val_w": val_w,
                    "val_y": val_y,
                }
            )

            b = candidates[-1]["boyce"]
            b_key = -np.inf if not np.isfinite(b) else float(b)
            if b_key > best_boyce_key + 1e-9:
                best_boyce_key = b_key
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    # select by Boyce, then AUC
    def key(c):
        b = c["boyce"]
        a = c["auc"]
        b_val = -np.inf if not np.isfinite(b) else float(b)
        a_val = -np.inf if not np.isfinite(a) else float(a)
        return (b_val, a_val)

    best = max(candidates, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | saved -> {ckpt_path}"
    )

    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": np.nan}
    return best_val, best["val_score01"], val_idx


# =========================================================
# Ensemble prediction (uLSIF)
# - each model outputs w(x) >= 0
# - ensemble = weighted mean of w
# - bounded score = w/(1+w)
# - optional normalization: divide by mean_q(w) so E_q[w]=1
# =========================================================
def ensemble_predict_on_indices_ulsif_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
    # If ULSIF_NORMALIZE_EQ1=True, pass q samples here (arrays, not indices):
    Xq_values: np.ndarray = None,
    Xq_masks: np.ndarray = None,
):
    in_value_channels = X_values.shape[1]

    models = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")

        model = CNNuLSIF(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            max_w=state.get("max_w", MAX_W),
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()
    weights_t = torch.tensor(w, dtype=torch.float32, device=device)

    # optional normalization constant
    norm_const = 1.0
    if ULSIF_NORMALIZE_EQ1:
        if Xq_values is None or Xq_masks is None:
            raise ValueError("ULSIF_NORMALIZE_EQ1=True requires Xq_values and Xq_masks (q-samples).")
        with torch.no_grad():
            xv_q = torch.from_numpy(Xq_values.astype(np.float32)).to(device)
            xm_q = torch.from_numpy(Xq_masks.astype(np.float32)).to(device)

            wq_ens = torch.zeros(xv_q.size(0), device=device)
            for k, model in enumerate(models):
                wq_ens += weights_t[k] * model(xv_q, xm_q)

            norm_const = float(wq_ens.mean().clamp_min(EPS_W).cpu())

    idx = np.asarray(indices, dtype=np.int64)
    all_score01, all_w, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            w_ens = torch.zeros(xv.size(0), device=device)
            for k, model in enumerate(models):
                w_ens += weights_t[k] * model(xv, xm)

            w_ens = w_ens / norm_const
            w_ens = torch.clamp(w_ens, 0.0, MAX_W)

            score01 = w_ens / (1.0 + w_ens)

            all_score01.append(score01.cpu().numpy())
            all_w.append(w_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return np.concatenate(all_score01), np.concatenate(all_w), np.concatenate(all_y)


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for metrics comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_boyces = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_ulsif_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_boyces.append(best_val["Boyce"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score = w/(1+w)):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold best Boyce:", fold_boyces)

    # ----------- Ensemble weights -----------
    # uLSIF loss is comparable across folds in principle, but to match your KLIEP setup we keep equal weights.
    weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
    print("\n[Ensemble] Using equal weights:", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_weights_equal.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_weights_equal.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    # If you want E_q[w]=1 normalization on test set, uncomment:
    # ULSIF_NORMALIZE_EQ1 = True
    # Xq_vals = X_test[y_test == Q_LABEL]
    # Xq_masks = M_test[y_test == Q_LABEL]

    test_score01, test_w, test_y = ensemble_predict_on_indices_ulsif_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
        # Xq_values=Xq_vals,
        # Xq_masks=Xq_masks,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score = w/(1+w)):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_w_ens.npy"), test_w)
    print("[Test] Saved test_score01.npy, test_w_ens.npy")


# Patch size: 65


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_65")
TEST_DIR = str(DATA_ROOT / "test_patches_65")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "ulsif" / "cnn_ulsif_patch_models_65")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 65  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Model selection gate (optional): computed on bounded monotone score = w/(1+w)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80

# uLSIF numeric stability
MAX_W = 1e3      # clamp w(x) to avoid blow-ups (tune if needed)
EPS_W = 1e-12

# Optional: normalize so E_q[w]=1 at inference time (NOT part of vanilla uLSIF)
ULSIF_NORMALIZE_EQ1 = False

# If your labels are reversed, flip these:
#   y==1 -> p (target)
#   y==0 -> q (reference)
P_LABEL = 1
Q_LABEL = 0


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers
# =========================================================
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    """Returns xv, xm, y. Useful for validation metrics only."""
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class PatchXDataset(Dataset):
    """Returns xv, xm only. Used for uLSIF p/q loaders."""
    def __init__(self, X_values, X_masks, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm


class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNuLSIF(nn.Module):
    """
    Neural uLSIF: network outputs w(x) >= 0 directly.
    Uses softplus for nonnegativity, plus clamp for stability.
    """
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, max_w=1e3):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.head = MLP(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)
        self.softplus = nn.Softplus(beta=1.0)
        self.max_w = max_w

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        raw = self.head(z)
        w = self.softplus(raw)  # >= 0
        if self.max_w is not None:
            w = torch.clamp(w, 0.0, self.max_w)
        return w


# =========================================================
# uLSIF loss
#   minimize 0.5 E_q[w^2] - E_p[w]
# =========================================================
def ulsif_loss(w_p: torch.Tensor, w_q: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, float]]:
    loss = 0.5 * (w_q ** 2).mean() - w_p.mean()
    stats = {
        "mean_w_p": float(w_p.detach().mean().cpu()),
        "mean_w_q": float(w_q.detach().mean().cpu()),
        "mean_wq2": float((w_q.detach() ** 2).mean().cpu()),
    }
    return loss, stats


# =========================================================
# Training per fold (uLSIF)
# - train on p/q batches
# - validation metrics computed on bounded score = w/(1+w)
# - returns OOF bounded score for that fold val split
# =========================================================
def train_and_save_fold_model_ulsif_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # p/q split inside train
    p_tr = train_idx[y_cv[train_idx] == P_LABEL]
    q_tr = train_idx[y_cv[train_idx] == Q_LABEL]
    p_va = val_idx[y_cv[val_idx] == P_LABEL]
    q_va = val_idx[y_cv[val_idx] == Q_LABEL]

    if len(p_tr) < 10 or len(q_tr) < 10:
        raise RuntimeError(f"[fold {fold_id}] too few samples: p_tr={len(p_tr)} q_tr={len(q_tr)}")

    print(f"\n[fold {fold_id}] train: p={len(p_tr)} q={len(q_tr)} | val: p={len(p_va)} q={len(q_va)}")

    ds_p_tr = PatchXDataset(X_values, X_masks, p_tr, train=True)
    ds_q_tr = PatchXDataset(X_values, X_masks, q_tr, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_p_tr = DataLoader(ds_p_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_q_tr = DataLoader(ds_q_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNuLSIF(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        max_w=MAX_W,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_train_epoch():
        model.train()
        total_loss = 0.0
        total_steps = 0
        mean_wq_avg = 0.0
        mean_wp_avg = 0.0

        for (xvp, xmp), (xvq, xmq) in zip(dl_p_tr, dl_q_tr):
            xvp, xmp = xvp.to(device), xmp.to(device)
            xvq, xmq = xvq.to(device), xmq.to(device)

            opt.zero_grad(set_to_none=True)
            w_p = model(xvp, xmp)
            w_q = model(xvq, xmq)
            loss, st = ulsif_loss(w_p, w_q)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += float(loss.detach().cpu())
            mean_wq_avg += st["mean_w_q"]
            mean_wp_avg += st["mean_w_p"]
            total_steps += 1

        return {
            "ulsif_loss": total_loss / max(1, total_steps),
            "mean_w_p": mean_wp_avg / max(1, total_steps),
            "mean_w_q": mean_wq_avg / max(1, total_steps),
        }

    def run_val_metrics():
        model.eval()
        all_w, all_y = [], []
        with torch.no_grad():
            for xv, xm, yb in dl_val:
                xv, xm = xv.to(device), xm.to(device)
                w = model(xv, xm)
                all_w.append(w.detach().cpu().numpy())
                all_y.append(yb.numpy())

        w_np = np.concatenate(all_w)
        y_np = np.concatenate(all_y).astype(int)

        # bounded monotone score in (0,1) for Boyce/AUC comparability
        score01 = w_np / (1.0 + w_np)
        mets = compute_boyce_and_auc(y_np, score01)
        return mets, score01, w_np, y_np

    candidates = []
    best_boyce_key = -np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr = run_train_epoch()
        va, val_score01, val_w, val_y = run_val_metrics()

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['ulsif_loss']:.4f}, mean_w_p {tr['mean_w_p']:.4f}, mean_w_q {tr['mean_w_q']:.4f} | "
            f"val AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "max_w": float(MAX_W),
                "p_label": int(P_LABEL),
                "q_label": int(Q_LABEL),
                "method": "uLSIF",
            }
            candidates.append(
                {
                    "epoch": ep,
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_score01": val_score01,
                    "val_w": val_w,
                    "val_y": val_y,
                }
            )

            b = candidates[-1]["boyce"]
            b_key = -np.inf if not np.isfinite(b) else float(b)
            if b_key > best_boyce_key + 1e-9:
                best_boyce_key = b_key
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    # select by Boyce, then AUC
    def key(c):
        b = c["boyce"]
        a = c["auc"]
        b_val = -np.inf if not np.isfinite(b) else float(b)
        a_val = -np.inf if not np.isfinite(a) else float(a)
        return (b_val, a_val)

    best = max(candidates, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | saved -> {ckpt_path}"
    )

    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": np.nan}
    return best_val, best["val_score01"], val_idx


# =========================================================
# Ensemble prediction (uLSIF)
# - each model outputs w(x) >= 0
# - ensemble = weighted mean of w
# - bounded score = w/(1+w)
# - optional normalization: divide by mean_q(w) so E_q[w]=1
# =========================================================
def ensemble_predict_on_indices_ulsif_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
    # If ULSIF_NORMALIZE_EQ1=True, pass q samples here (arrays, not indices):
    Xq_values: np.ndarray = None,
    Xq_masks: np.ndarray = None,
):
    in_value_channels = X_values.shape[1]

    models = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_ulsif_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")

        model = CNNuLSIF(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            max_w=state.get("max_w", MAX_W),
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()
    weights_t = torch.tensor(w, dtype=torch.float32, device=device)

    # optional normalization constant
    norm_const = 1.0
    if ULSIF_NORMALIZE_EQ1:
        if Xq_values is None or Xq_masks is None:
            raise ValueError("ULSIF_NORMALIZE_EQ1=True requires Xq_values and Xq_masks (q-samples).")
        with torch.no_grad():
            xv_q = torch.from_numpy(Xq_values.astype(np.float32)).to(device)
            xm_q = torch.from_numpy(Xq_masks.astype(np.float32)).to(device)

            wq_ens = torch.zeros(xv_q.size(0), device=device)
            for k, model in enumerate(models):
                wq_ens += weights_t[k] * model(xv_q, xm_q)

            norm_const = float(wq_ens.mean().clamp_min(EPS_W).cpu())

    idx = np.asarray(indices, dtype=np.int64)
    all_score01, all_w, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            w_ens = torch.zeros(xv.size(0), device=device)
            for k, model in enumerate(models):
                w_ens += weights_t[k] * model(xv, xm)

            w_ens = w_ens / norm_const
            w_ens = torch.clamp(w_ens, 0.0, MAX_W)

            score01 = w_ens / (1.0 + w_ens)

            all_score01.append(score01.cpu().numpy())
            all_w.append(w_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return np.concatenate(all_score01), np.concatenate(all_w), np.concatenate(all_y)


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for metrics comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_boyces = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_ulsif_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_boyces.append(best_val["Boyce"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score = w/(1+w)):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold best Boyce:", fold_boyces)

    # ----------- Ensemble weights -----------
    # uLSIF loss is comparable across folds in principle, but to match your KLIEP setup we keep equal weights.
    weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
    print("\n[Ensemble] Using equal weights:", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_weights_equal.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_weights_equal.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    # If you want E_q[w]=1 normalization on test set, uncomment:
    # ULSIF_NORMALIZE_EQ1 = True
    # Xq_vals = X_test[y_test == Q_LABEL]
    # Xq_masks = M_test[y_test == Q_LABEL]

    test_score01, test_w, test_y = ensemble_predict_on_indices_ulsif_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
        # Xq_values=Xq_vals,
        # Xq_masks=Xq_masks,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score = w/(1+w)):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_w_ens.npy"), test_w)
    print("[Test] Saved test_score01.npy, test_w_ens.npy")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

path = str(OUTPUT_ROOT / "Boyce_index.csv")

df = pd.read_csv(path, skiprows=1)
df.columns = df.columns.str.strip()

print(df.columns)



In [ ]:
plt.figure()

plt.plot(df["patch size"], df["kliep"], marker="o", label="KLIEP")
plt.plot(df["patch size"], df["classification"], marker="o", label="Classification")
plt.plot(df["patch size"], df["ulsif"], marker="o", label="uLSIF")
plt.plot(df["patch size"], df["kulsif"], marker="o", label="KuLSIF")

plt.xlabel("Patch size")
plt.ylabel("Boyce index")
plt.legend()
plt.tight_layout()
plt.show()
